In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into memory (keep full dataframes in kernel variables)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Ensure timestamp is integer-like and add an Asia/Shanghai timezone-aware datetime column (UTC+8) for potential later use
tz_sh = pytz.timezone('Asia/Shanghai')
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        # coerce to integer if possible
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        # create a localized datetime column without shifting (treat timestamps as UTC+8 per rule)
        df['dt_sh'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce').dt.tz_localize(tz_sh)

# 1) For each file: number of rows and min/max timestamp (raw values). One line per file.
summary_counts = pd.DataFrame([
    {'file': 'metric.csv', 'rows': int(metric_df.shape[0]), 'min_timestamp': int(metric_df['timestamp'].min()) if metric_df['timestamp'].notna().any() else None, 'max_timestamp': int(metric_df['timestamp'].max()) if metric_df['timestamp'].notna().any() else None},
    {'file': 'trace.csv',  'rows': int(trace_df.shape[0]),  'min_timestamp': int(trace_df['timestamp'].min()) if trace_df['timestamp'].notna().any() else None, 'max_timestamp': int(trace_df['timestamp'].max()) if trace_df['timestamp'].notna().any() else None},
    {'file': 'log.csv',    'rows': int(log_df.shape[0]),    'min_timestamp': int(log_df['timestamp'].min()) if log_df['timestamp'].notna().any() else None, 'max_timestamp': int(log_df['timestamp'].max()) if log_df['timestamp'].notna().any() else None},
    {'file': 'error_logs.csv','rows': int(error_logs_df.shape[0]), 'min_timestamp': int(error_logs_df['timestamp'].min()) if error_logs_df['timestamp'].notna().any() else None, 'max_timestamp': int(error_logs_df['timestamp'].max()) if error_logs_df['timestamp'].notna().any() else None},
])

# 2) From metric.csv: distinct kpi_name values and top 20 cmdb_id by row count.
metric_kpi_names = pd.Series(sorted(metric_df['kpi_name'].dropna().unique())).to_frame(name='kpi_name')
# If more than 20 distinct kpi_name, return first 50 alphabetically per instruction
if metric_kpi_names.shape[0] > 20:
    metric_kpi_names_disp = metric_kpi_names.head(50)
else:
    metric_kpi_names_disp = metric_kpi_names

metric_top_cmdb = metric_df.groupby('cmdb_id', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 3) From trace.csv: distinct trace_name values and top 20 cmdb_id by row count.
trace_names = pd.Series(sorted(trace_df['trace_name'].dropna().unique())).to_frame(name='trace_name')
trace_names_disp = trace_names.head(20)  # limit to 20 as requested
trace_top_cmdb = trace_df.groupby('cmdb_id', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 4) From log.csv: distinct log_name values and top 20 cmdb_id by row count.
log_names = pd.Series(sorted(log_df['log_name'].dropna().unique())).to_frame(name='log_name')
log_names_disp = log_names.head(20)
log_top_cmdb = log_df.groupby('cmdb_id', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# 5) From error_logs.csv: top 20 cmdb_id by row count and up to 5 sample messages (one per cmdb_id) for the top 5 cmdb_ids.
error_top20 = error_logs_df.groupby('cmdb_id', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20)
top5_cmdbs = error_top20['cmdb_id'].head(5).tolist()

# For each of top5 cmdbs, get one sample message (if exists)
sample_rows = []
for cm in top5_cmdbs:
    subset = error_logs_df[error_logs_df['cmdb_id'] == cm]
    if not subset.empty:
        sample_msg = subset['message'].dropna().iloc[0] if subset['message'].dropna().shape[0] > 0 else None
    else:
        sample_msg = None
    sample_rows.append({'cmdb_id': cm, 'sample_message': sample_msg})
error_top5_samples = pd.DataFrame(sample_rows)

# Display compact summaries (limited rows)
summary_counts, \
metric_kpi_names_disp, metric_top_cmdb[['cmdb_id','count']], \
trace_names_disp, trace_top_cmdb[['cmdb_id','count']], \
log_names_disp, log_top_cmdb[['cmdb_id','count']], \
error_top20[['cmdb_id','count']], error_top5_samples

```
Out[1]:
```
```python
# Summarize the previously computed results in plain English (compact).
# Uses variables from prior cell: summary_counts, metric_kpi_names, metric_top_cmdb,
# trace_names, trace_top_cmdb, log_names, log_top_cmdb, error_top20, error_top5_samples

# Build compact summaries
file_lines = []
for _, r in summary_counts.iterrows():
    file_lines.append(f"{r['file']}: rows={int(r['rows'])}, min_timestamp={int(r['min_timestamp']) if pd.notna(r['min_timestamp']) else 'N/A'}, max_timestamp={int(r['max_timestamp']) if pd.notna(r['max_timestamp']) else 'N/A'}")

# Metric KPI summary
metric_kpi_count = metric_kpi_names.shape[0]
metric_kpi_sample = metric_kpi_names['kpi_name'].head(10).tolist()
metric_top20_list = [(row['cmdb_id'], int(row['count'])) for _, row in metric_top_cmdb.iterrows()]

# Trace summary
trace_name_count = trace_names.shape[0]
trace_name_sample = trace_names['trace_name'].head(10).tolist()
trace_top20_list = [(row['cmdb_id'], int(row['count'])) for _, row in trace_top_cmdb.iterrows()]

# Log summary
log_name_count = log_names.shape[0]
log_name_sample = log_names['log_name'].head(10).tolist()
log_top20_list = [(row['cmdb_id'], int(row['count'])) for _, row in log_top_cmdb.iterrows()]

# Error logs summary
error_total_rows = int(error_logs_df.shape[0])
error_top20_list = [(row['cmdb_id'], int(row['count'])) for _, row in error_top20.iterrows()]
error_samples = []
for _, row in error_top5_samples.iterrows():
    error_samples.append((row['cmdb_id'], row['sample_message']))

# Compose summary string
summary = (
    "File summaries (rows, min_timestamp, max_timestamp):\n  - " + "\n  - ".join(file_lines) +
    "\n\nMetric (metric.csv):\n"
    f"  - distinct kpi_name count: {metric_kpi_count}\n"
    f"  - sample kpi_name (first 10 alphabetical): {metric_kpi_sample}\n"
    f"  - top 20 cmdb_id by row count (cmdb_id, count): {metric_top20_list}\n\n"
    "Trace (trace.csv):\n"
    f"  - distinct trace_name count: {trace_name_count}\n"
    f"  - sample trace_name (first 10 alphabetical): {trace_name_sample}\n"
    f"  - top 20 cmdb_id by row count (cmdb_id, count): {trace_top20_list}\n\n"
    "Log (log.csv):\n"
    f"  - distinct log_name count: {log_name_count}\n"
    f"  - log_name values: {log_name_sample}\n"
    f"  - top 20 cmdb_id by row count (cmdb_id, count): {log_top20_list}\n\n"
    "Error Logs (error_logs.csv):\n"
    f"  - total rows: {error_total_rows}\n"
    f"  - top 20 cmdb_id by row count (cmdb_id, count): {error_top20_list}\n"
    f"  - up to 5 sample messages for top 5 cmdb_ids: {error_samples}\n\n"
    "Notes: timestamps shown are raw Unix seconds. error_logs.csv contains no rows, so no sample messages exist."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file    rows  min_timestamp  max_timestamp
0      metric.csv  321412   1.647788e+09   1.647792e+09
1       trace.csv   24656   1.647790e+09   1.647792e+09
2         log.csv    1670   1.647790e+09   1.647792e+09
3  error_logs.csv       0            NaN            NaN,                                              kpi_name
0                                      app.grpc.count
1                                        app.grpc.mrt
2                                         app.grpc.rr
3                                         app.grpc.sr
4                                      app.http.count
..                                                ...
45             container.node-5.container_memory_swap
46         container.node-5.container_memory_usage_MB
47   container.node-5.container_memory_working_set_MB
48  container.node-5.container_network_receive_MB....
49  container.node-5.container_network_receive_err...

[50 rows x 1 columns],                      cmdb_id  count
0                  adservice  20801
4                 adservice2  20313
30               frontend2-0  11956
29                frontend-2  11468
28                frontend-1  11468
27                frontend-0  11468
13         checkoutservice-1   8784
15        checkoutservice2-0   8784
14         checkoutservice-2   8540
12         checkoutservice-0   8540
44   productcatalogservice-1   7320
43   productcatalogservice-0   7320
45   productcatalogservice-2   7076
46  productcatalogservice2-0   6344
51  recommendationservice2-0   6222
9              cartservice-2   6100
7              cartservice-0   6100
10            cartservice2-0   6100
49   recommendationservice-1   5978
50   recommendationservice-2   5978,                                      trace_name
0    trace.from_checkoutservice-0.duration_mean
1     trace.from_checkoutservice-0.duration_p95
2       trace.from_checkoutservice-0.error_rate
3        trace.from_checkoutservice-0.row_count
4    trace.from_checkoutservice-1.duration_mean
5     trace.from_checkoutservice-1.duration_p95
6       trace.from_checkoutservice-1.error_rate
7        trace.from_checkoutservice-1.row_count
8    trace.from_checkoutservice-2.duration_mean
9     trace.from_checkoutservice-2.duration_p95
10      trace.from_checkoutservice-2.error_rate
11       trace.from_checkoutservice-2.row_count
12  trace.from_checkoutservice2-0.duration_mean
13   trace.from_checkoutservice2-0.duration_p95
14     trace.from_checkoutservice2-0.error_rate
15      trace.from_checkoutservice2-0.row_count
16          trace.from_frontend-0.duration_mean
17           trace.from_frontend-0.duration_p95
18             trace.from_frontend-0.error_rate
19              trace.from_frontend-0.row_count,                      cmdb_id  count
21                frontend-2   2608
20                frontend-0   2548
9          checkoutservice-1   1508
8          checkoutservice-0   1496
10         checkoutservice-2   1460
22               frontend2-0   1056
11        checkoutservice2-0    972
28   productcatalogservice-1    940
29   productcatalogservice-2    936
27   productcatalogservice-0    932
31   recommendationservice-0    720
33   recommendationservice-2    720
32   recommendationservice-1    720
34  recommendationservice2-0    600
4              cartservice-0    584
6              cartservice-2    576
5              cartservice-1    576
12         currencyservice-0    508
13         currencyservice-1    508
14         currencyservice-2    508,           log_name
0  log.error_count
1    log.row_count,                     cmdb_id  count
4             cartservice-1     62
5             cartservice-2     62
10        currencyservice-1     62
9         currencyservice-0     62
11        currencyservice-2     62
29        shippingservice-1     62
15               frontend-0     62
25  recommendationservice-1     62
30        shippingservice-2     62
17               frontend-2     60
23  productcatalogservice-2     60
1               adservice-1     60
3             cartservice-0     60
2               adservice-2     60
0               adservice-0     60
26  recommendationservice-2     60
24  recommendationservice-0     60
28        shippingservice-0     60
22  productcatalogservice-1     60
21  productcatalogservice-0     60, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: []
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df from kernel memory
# 1) Compute global thresholds (p95 and p5) per (cmdb_id, kpi_name)
threshold_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# 2) Filter metric rows into the incident window (UTC+8): 
# March 20, 2022 23:30 to March 21, 2022 00:00 inclusive.
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz_sh).timestamp())
end_ts   = int(pd.Timestamp('2022-03-21 00:00:00', tz=tz_sh).timestamp())

filtered_metrics = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into filtered_metrics and mark anomalies
filtered_metrics = filtered_metrics.merge(threshold_df, on=['cmdb_id','kpi_name'], how='left')

# boolean flags
filtered_metrics['high_anomaly'] = filtered_metrics['value'] >= filtered_metrics['p95']
filtered_metrics['low_anomaly']  = filtered_metrics['value'] <= filtered_metrics['p5']

# relative severity columns (guard against division by zero)
filtered_metrics['rel_high'] = np.where(
    (filtered_metrics['high_anomaly']) & (filtered_metrics['p95'] != 0),
    (filtered_metrics['value'] - filtered_metrics['p95']) / filtered_metrics['p95'],
    np.nan
)
filtered_metrics['rel_low'] = np.where(
    (filtered_metrics['low_anomaly']) & (filtered_metrics['p5'] != 0),
    (filtered_metrics['p5'] - filtered_metrics['value']) / filtered_metrics['p5'],
    np.nan
)

# 4a) Count of unique (cmdb_id,kpi_name) pairs in threshold_df
unique_pairs_count = int(threshold_df.shape[0])

# Prepare anomalies subset for grouping
anomalies = filtered_metrics[filtered_metrics['high_anomaly'] | filtered_metrics['low_anomaly']].copy()

# 4b) Number of pairs that have at least one high_anomaly and number with at least one low_anomaly
high_pairs = anomalies[anomalies['high_anomaly']].groupby(['cmdb_id','kpi_name']).size().reset_index(name='cnt')
low_pairs  = anomalies[anomalies['low_anomaly']].groupby(['cmdb_id','kpi_name']).size().reset_index(name='cnt')

high_pairs_count = int(high_pairs.shape[0])
low_pairs_count  = int(low_pairs.shape[0])

# 4c) Top 20 (cmdb_id,kpi_name) pairs ranked by total anomaly count with required fields
if anomalies.empty:
    top20_pairs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','anomaly_type','max_relative_severity'
    ])
else:
    grp = anomalies.groupby(['cmdb_id','kpi_name'], as_index=False)
    pair_stats = grp.agg(
        anomaly_count=('value','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        high_count=('high_anomaly','sum'),
        low_count=('low_anomaly','sum'),
        rel_high_max=('rel_high','max'),
        rel_low_max=('rel_low','max')
    )
    # determine anomaly_type
    def decide_type(row):
        if row['high_count'] > row['low_count']:
            return 'high'
        elif row['low_count'] > row['high_count']:
            return 'low'
        else:
            return 'mixed'
    pair_stats['anomaly_type'] = pair_stats.apply(decide_type, axis=1)
    # compute max_relative_severity per spec
    pair_stats['max_relative_severity'] = np.where(
        pair_stats['high_count'] > 0,
        pair_stats['rel_high_max'],
        pair_stats['rel_low_max']
    )
    # compact numeric columns
    pair_stats['max_relative_severity'] = pair_stats['max_relative_severity'].replace([np.inf,-np.inf], np.nan)
    pair_stats['max_relative_severity'] = pair_stats['max_relative_severity'].round(6)
    pair_stats['earliest_anomaly_timestamp'] = pair_stats['earliest_anomaly_timestamp'].astype('Int64')
    # select and sort
    top20_pairs_df = pair_stats.sort_values(['anomaly_count','earliest_anomaly_timestamp'], ascending=[False,True])[
        ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','anomaly_type','max_relative_severity']
    ].head(20).reset_index(drop=True)

# 4d) Top 10 cmdb_id by total anomaly count across all their KPIs
if anomalies.empty:
    top10_components_df = pd.DataFrame(columns=['cmdb_id','total_anomalies','number_of_kpis_with_anomalies'])
else:
    comp_stats = anomalies.groupby('cmdb_id').agg(
        total_anomalies=('value','size'),
        number_of_kpis_with_anomalies=('kpi_name', pd.Series.nunique)
    ).reset_index()
    top10_components_df = comp_stats.sort_values('total_anomalies', ascending=False).head(10).reset_index(drop=True)

# Ensure threshold_df and filtered_metrics (with anomaly flags) are stored in kernel variables for next steps
# threshold_df, filtered_metrics are available.

# Return only the requested compact aggregates
unique_pairs_count, high_pairs_count, low_pairs_count, top20_pairs_df, top10_components_df

```
Out[2]:
```
```python
# Summarize results in plain English using existing kernel variables:
# unique_pairs_count, high_pairs_count, low_pairs_count, top20_pairs_df, top10_components_df, threshold_df, filtered_metrics

examples = []
for r in top20_pairs_df.head(5).itertuples(index=False):
    examples.append(f"({r.cmdb_id}, {r.kpi_name}, count={int(r.anomaly_count)}, ts={int(r.earliest_anomaly_timestamp)}, type={r.anomaly_type})")

summary = (
    f"Summary of anomaly analysis (incident window: 2022-03-20 23:30 to 2022-03-21 00:00 UTC+8):\n"
    f"- Unique (cmdb_id, kpi_name) pairs analyzed: {int(unique_pairs_count)}\n"
    f"- Pairs with ≥1 high anomaly in window: {int(high_pairs_count)}\n"
    f"- Pairs with ≥1 low anomaly in window: {int(low_pairs_count)}\n\n"
    f"- Top examples of (cmdb_id, kpi_name, anomaly_count, earliest_timestamp, anomaly_type):\n"
    f"  - " + "; ".join(examples) + "\n\n"
    f"- Top 10 components by total anomaly count (component, total_anomalies, number_of_kpis_with_anomalies):\n"
    + "\n  - ".join([
        f"{row.cmdb_id}: total_anomalies={int(row.total_anomalies)}, kpis_with_anomalies={int(row.number_of_kpis_with_anomalies)}"
        for row in top10_components_df.itertuples(index=False)
      ])
    + "\n\n"
    f"Notes:\n"
    f"- Timestamps shown are raw Unix seconds (UTC+8 context).\n"
    f"- Many top KPI anomalies are from 'adservice' (counts ~31 per KPI in top list) and are labeled mostly 'mixed'.\n"
    f"- threshold_df and filtered_metrics (with high_anomaly/low_anomaly flags and relative severities) are saved in kernel variables for next steps."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(5282, 5119, 5076,       cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp anomaly_type  max_relative_severity
0   adservice                                        app.grpc.rr             31                  1647790200        mixed                    0.0
1   adservice                                        app.grpc.sr             31                  1647790200        mixed                    0.0
2   adservice                                     app.http.count             31                  1647790200        mixed                    0.0
3   adservice                                        app.http.rr             31                  1647790200        mixed                    0.0
4   adservice                                        app.http.sr             31                  1647790200        mixed                    0.0
5   adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31                  1647790200        mixed                    0.0
6   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             31                  1647790200        mixed                    0.0
7   adservice  runtime.java_lang_ClassLoading_UnloadedClassCount             31                  1647790200        mixed                    0.0
8   adservice             runtime.java_lang_ClassLoading_Verbose             31                  1647790200        mixed                    NaN
9   adservice  runtime.java_lang_Compilation_CompilationTimeM...             31                  1647790200        mixed                    0.0
10  adservice  runtime.java_lang_Compilation_TotalCompilation...             31                  1647790200        mixed                    0.0
11  adservice  runtime.java_lang_GarbageCollector_CollectionC...             31                  1647790200        mixed                    0.0
12  adservice  runtime.java_lang_GarbageCollector_CollectionT...             31                  1647790200        mixed                    0.0
13  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0
14  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0
15  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0
16  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0
17  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             31                  1647790200        mixed                    0.0,               cmdb_id  total_anomalies  number_of_kpis_with_anomalies
0           adservice             9674                            341
1          adservice2             9470                            333
2          frontend-1             5613                            188
3         frontend2-0             4399                            196
4          frontend-2             4242                            188
5          frontend-0             4198                            188
6   checkoutservice-0             3192                            140
7   checkoutservice-1             3103                            144
8   checkoutservice-2             3097                            136
9  checkoutservice2-0             3086                            144)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: filtered_metrics, start_ts, end_ts
# Ensure start_ts and end_ts exist (if not, recreate for the incident window in UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
try:
    start_ts, end_ts
except NameError:
    start_ts = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz_sh).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-21 00:00:00', tz=tz_sh).timestamp())

# Work on anomaly rows only
anoms = filtered_metrics[(filtered_metrics['high_anomaly'] | filtered_metrics['low_anomaly'])].copy()
# If empty, prepare empty faults_df with expected columns
if anoms.empty:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes',
        'anomaly_types_in_run','max_relative_severity','extremal_value',
        'high_count','low_count'
    ])
else:
    # Sort and detect consecutive runs (sampling interval = 60 seconds)
    anoms.sort_values(['cmdb_id','kpi_name','timestamp'], inplace=True)
    anoms['ts_diff'] = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().fillna(0).astype('Int64')
    anoms['new_run'] = (anoms['ts_diff'] != 60)
    # run index per (cmdb_id,kpi_name)
    anoms['run_idx'] = anoms.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum().astype('Int64')

    # Aggregate per run
    run_grp = anoms.groupby(['cmdb_id','kpi_name','run_idx'], as_index=False)
    runs = run_grp.agg(
        run_start_ts=('timestamp','min'),
        run_end_ts=('timestamp','max'),
        run_length_minutes=('timestamp','size'),
        high_count=('high_anomaly','sum'),
        low_count=('low_anomaly','sum'),
        rel_high_max=('rel_high','max'),
        rel_low_max=('rel_low','max'),
        max_value=('value','max'),
        min_value=('value','min')
    )

    # Determine anomaly_types_in_run
    def run_type(r):
        if r['high_count'] > 0 and r['low_count'] == 0:
            return 'high'
        elif r['low_count'] > 0 and r['high_count'] == 0:
            return 'low'
        elif r['high_count'] > 0 and r['low_count'] > 0:
            return 'mixed'
        else:
            return 'unknown'
    runs['anomaly_types_in_run'] = runs.apply(run_type, axis=1)

    # max_relative_severity: maximum across rel_high and rel_low (ignore NaN)
    runs['max_relative_severity'] = runs[['rel_high_max','rel_low_max']].max(axis=1)
    runs['max_relative_severity'] = runs['max_relative_severity'].replace([np.inf, -np.inf], np.nan).round(6)

    # extremal_value: choose max for high-dominated runs, min for low-dominated runs, for mixed choose max if high_count>low_count else min if low_count>high_count else NaN
    def extremal(r):
        if r['anomaly_types_in_run'] == 'high':
            return r['max_value']
        elif r['anomaly_types_in_run'] == 'low':
            return r['min_value']
        elif r['anomaly_types_in_run'] == 'mixed':
            if r['high_count'] > r['low_count']:
                return r['max_value']
            elif r['low_count'] > r['high_count']:
                return r['min_value']
            else:
                return np.nan
        else:
            return np.nan
    runs['extremal_value'] = runs.apply(extremal, axis=1)

    # Final faults_df with required columns (keep high_count/low_count for later aggregation but they can be hidden in displays)
    faults_df = runs[[
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes',
        'anomaly_types_in_run','max_relative_severity','extremal_value','high_count','low_count'
    ]].copy()
    # ensure integer types where appropriate
    faults_df['run_start_ts'] = faults_df['run_start_ts'].astype('Int64')
    faults_df['run_end_ts'] = faults_df['run_end_ts'].astype('Int64')
    faults_df['run_length_minutes'] = faults_df['run_length_minutes'].astype('Int64')

# 2a) Total number of fault runs
total_runs = int(faults_df.shape[0])

# 2b) Number of fault runs that start inside the incident window
runs_start_in_window = int(faults_df[(faults_df['run_start_ts'] >= start_ts) & (faults_df['run_start_ts'] <= end_ts)].shape[0])

# 2c) Top 20 fault runs ranked by max_relative_severity
if faults_df.empty:
    top20_faults_df = faults_df.head(0)
else:
    top20_faults_df = faults_df.sort_values('max_relative_severity', ascending=False).head(20)[
        ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity']
    ].reset_index(drop=True)

# 2d) Top 20 components ranked by number_of_fault_runs and earliest run_start_ts
if faults_df.empty:
    top20_components_df = pd.DataFrame(columns=['cmdb_id','number_of_runs','number_of_distinct_kpis_with_runs','earliest_run_start_ts'])
else:
    comp_stats = faults_df.groupby('cmdb_id').agg(
        number_of_runs=('kpi_name','size'),
        number_of_distinct_kpis_with_runs=('kpi_name', pd.Series.nunique),
        earliest_run_start_ts=('run_start_ts','min')
    ).reset_index()
    top20_components_df = comp_stats.sort_values(['number_of_runs','earliest_run_start_ts'], ascending=[False,True]).head(20).reset_index(drop=True)
    top20_components_df['earliest_run_start_ts'] = top20_components_df['earliest_run_start_ts'].astype('Int64')

# 2e) For top 10 components by number_of_runs, earliest fault run_start_ts and dominant anomaly type
if faults_df.empty:
    top10_component_dominant_df = pd.DataFrame(columns=['cmdb_id','earliest_run_start_ts','dominant_anomaly_type','number_of_runs'])
else:
    top10 = top20_components_df.head(10)['cmdb_id'].tolist()
    comp_run_types = faults_df[faults_df['cmdb_id'].isin(top10)].copy()
    # count run types per component
    comp_dom = comp_run_types.groupby(['cmdb_id']).agg(
        total_runs=('kpi_name','size'),
        high_runs=('anomaly_types_in_run', lambda s: (s=='high').sum()),
        low_runs=('anomaly_types_in_run', lambda s: (s=='low').sum()),
        mixed_runs=('anomaly_types_in_run', lambda s: (s=='mixed').sum()),
        earliest_run_start_ts=('run_start_ts','min')
    ).reset_index()
    # dominant: >50% high -> high, >50% low -> low, else mixed
    def dominant(row):
        if row['high_runs'] > 0.5 * row['total_runs']:
            return 'high'
        elif row['low_runs'] > 0.5 * row['total_runs']:
            return 'low'
        else:
            return 'mixed'
    comp_dom['dominant_anomaly_type'] = comp_dom.apply(dominant, axis=1)
    top10_component_dominant_df = comp_dom[['cmdb_id','earliest_run_start_ts','dominant_anomaly_type','total_runs']].rename(columns={'total_runs':'number_of_runs'})
    top10_component_dominant_df['earliest_run_start_ts'] = top10_component_dominant_df['earliest_run_start_ts'].astype('Int64')

# Save faults_df in kernel for next steps (already assigned)

# Return compact aggregates only
total_runs, runs_start_in_window, top20_faults_df, top20_components_df, top10_component_dominant_df, faults_df.shape

```
Out[3]:
```
```python
# Plain-English summary of the fault identification results (compact).
summary = (
    "Fault identification summary (UTC+8):\n"
    "- Total fault runs detected: 10,438\n"
    "- Fault runs that start inside the incident window: 10,438\n\n"
    "- Earliest run start (most components): 1647790200 (2022-03-20 23:30:00 UTC+8)\n\n"
    "- Top example fault runs by max_relative_severity (cmdb_id, kpi_name, run_start_ts, run_end_ts, run_length_minutes, type, max_relative_severity):\n"
    "  - paymentservice-2, container.node-6.container_cpu_system_seconds, 1647790200-1647792000, 31 min, mixed, 1.1259e+13\n"
    "  - productcatalogservice-0, container.node-6.container_fs_reads./dev/vda, 1647790200-1647792000, 31 min, mixed, 632.0\n"
    "  - node-1, system.net.tcp.retrans_segs, 1647790980-1647791580, 11 min, mixed, 233.3333\n"
    "  - productcatalogservice-0, container.node-6.container_fs_reads_MB./dev/vda, 1647790200-1647792000, 31 min, mixed, 135.6087\n"
    "  - frontend-1, container.node-5.container_cpu_cfs_periods, 1647791700-1647792000, 6 min, mixed, 45.25\n\n"
    "- Top components by number of fault runs (component: number_of_runs, distinct_kpis_with_runs, earliest_run_start_ts):\n"
    "  1) checkoutservice-2: 509 runs, 136 kpis, earliest 1647790200\n"
    "  2) checkoutservice-0: 454 runs, 140 kpis, earliest 1647790200\n"
    "  3) frontend-2: 431 runs, 188 kpis, earliest 1647790200\n"
    "  4) adservice: 394 runs, 341 kpis, earliest 1647790200\n"
    "  5) adservice2: 382 runs, 333 kpis, earliest 1647790200\n"
    "  6) frontend-0: 370 runs, 188 kpis, earliest 1647790200\n"
    "  7) checkoutservice-1: 364 runs, 144 kpis, earliest 1647790200\n"
    "  8) checkoutservice2-0: 360 runs, 144 kpis, earliest 1647790200\n"
    "  9) frontend2-0: 343 runs, 196 kpis, earliest 1647790200\n"
    " 10) productcatalogservice-2: 261 runs, 116 kpis, earliest 1647790200\n\n"
    "- Dominant anomaly type for top 10 components (by number_of_runs):\n"
    "  - Most components are 'mixed' (no single type >50% of runs).\n"
    "  - Note: checkoutservice-2 is 'low' dominated (majority low-run types).\n\n"
    "Notes:\n"
    "- faults_df (per-run records) is saved in the kernel for follow-up trace/log analysis.\n"
    "- Numeric severities include extremely large values for some metrics (e.g., paymentservice-2), and many mixed-type runs across services.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(10438, 10438,                     cmdb_id                                           kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity
0          paymentservice-2      container.node-6.container_cpu_system_seconds    1647790200  1647792000                  31                mixed           1.125900e+13
1   productcatalogservice-0       container.node-6.container_fs_reads./dev/vda    1647790200  1647792000                  31                mixed           6.320000e+02
2                    node-1                        system.net.tcp.retrans_segs    1647790980  1647791580                  11                mixed           2.333333e+02
3   productcatalogservice-0    container.node-6.container_fs_reads_MB./dev/vda    1647790200  1647792000                  31                mixed           1.356087e+02
4                frontend-1         container.node-5.container_cpu_cfs_periods    1647791700  1647792000                   6                mixed           4.525000e+01
5                frontend-1       container.node-5.container_cpu_usage_seconds    1647792000  1647792000                   1                 high           4.017519e+01
6         checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...    1647791280  1647792000                  13                mixed           2.259524e+01
7          paymentservice-0  container.node-6.container_network_receive_MB....    1647790680  1647790680                   1                 high           2.181847e+01
8            emailservice-0  container.node-6.container_network_receive_MB....    1647791460  1647791460                   1                 high           2.098265e+01
9           emailservice2-0  container.node-6.container_network_receive_MB....    1647791160  1647791160                   1                 high           1.775518e+01
10        shippingservice-2  container.node-5.container_network_receive_MB....    1647791040  1647791040                   1                 high           1.736907e+01
11               frontend-1      container.node-5.container_cpu_system_seconds    1647791280  1647792000                  13                mixed           1.700000e+01
12                   node-2                                  system.io.w_await    1647790500  1647790620                   3                mixed           1.670115e+01
13                   node-2                                    system.io.await    1647790500  1647790620                   3                mixed           1.670115e+01
14                   node-1                                  system.cpu.iowait    1647790200  1647790200                   1                 high           1.462000e+01
15                   node-5                                    system.io.rkb_s    1647790200  1647791940                  30                mixed           1.406250e+01
16        checkoutservice-2  container.node-5.container_network_receive_MB....    1647791220  1647791220                   1                 high           1.367191e+01
17                   node-1                                      system.io.w_s    1647791400  1647791400                   1                 high           1.358491e+01
18              adservice-0  container.node-6.container_network_receive_MB....    1647791880  1647791880                   1                 high           1.294246e+01
19                   node-4                              system.net.bytes_rcvd    1647791760  1647791760                   1                 high           1.254458e+01,                      cmdb_id  number_of_runs  number_of_distinct_kpis_with_runs  earliest_run_start_ts
0          checkoutservice-2             509                                136             1647790200
1          checkoutservice-0             454                                140             1647790200
2                 frontend-2             431                                188             1647790200
3                  adservice             394                                341             1647790200
4                 adservice2             382                                333             1647790200
5                 frontend-0             370                                188             1647790200
6          checkoutservice-1             364                                144             1647790200
7         checkoutservice2-0             360                                144             1647790200
8                frontend2-0             343                                196             1647790200
9    productcatalogservice-2             261                                116             1647790200
10   productcatalogservice-0             239                                120             1647790200
11   productcatalogservice-1             226                                120             1647790200
12                frontend-1             211                                188             1647790200
13             cartservice-2             198                                100             1647790200
14                    node-4             196                                 59             1647790200
15  productcatalogservice2-0             190                                104             1647790200
16         shippingservice-1             187                                 80             1647790200
17             cartservice-1             183                                 96             1647790200
18                    node-3             181                                 58             1647790200
19        shippingservice2-0             179                                 80             1647790200,                    cmdb_id  earliest_run_start_ts dominant_anomaly_type  number_of_runs
0                adservice             1647790200                 mixed             394
1               adservice2             1647790200                 mixed             382
2        checkoutservice-0             1647790200                 mixed             454
3        checkoutservice-1             1647790200                 mixed             364
4        checkoutservice-2             1647790200                   low             509
5       checkoutservice2-0             1647790200                 mixed             360
6               frontend-0             1647790200                 mixed             370
7               frontend-2             1647790200                 mixed             431
8              frontend2-0             1647790200                 mixed             343
9  productcatalogservice-2             1647790200                 mixed             261, (10438, 10))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df and timezone/start/end variables if present
tz_sh = pytz.timezone('Asia/Shanghai')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamps and values are numeric
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Ensure incident window timestamps exist (reuse start_ts/end_ts if earlier)
try:
    start_ts, end_ts
except NameError:
    start_ts = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz_sh).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-21 00:00:00', tz=tz_sh).timestamp())

# 1) Compute global thresholds (p95 and p5) per (cmdb_id, trace_name)
trace_threshold_df = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# 2) Filter trace.csv to incident window
filtered_traces = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Mark anomalies per rules using thresholds (merge thresholds)
filtered_traces = filtered_traces.merge(trace_threshold_df, on=['cmdb_id','trace_name'], how='left')

# Identify trace kind by name
tn = filtered_traces['trace_name'].astype(str)
is_duration = tn.str.contains('duration', case=False, na=False)
is_error_rate = tn.str.contains('error_rate', case=False, na=False)
is_row_count = tn.str.contains('row_count', case=False, na=False)

# Initialize flags
filtered_traces['high_anomaly'] = False
filtered_traces['low_anomaly']  = False
filtered_traces['anomaly_kind'] = None

# Apply rules
# duration -> high if value >= p95
mask = is_duration & (filtered_traces['value'] >= filtered_traces['p95'])
filtered_traces.loc[mask, 'high_anomaly'] = True
filtered_traces.loc[mask, 'anomaly_kind'] = 'duration'

# error_rate -> high if value >= p95
mask = is_error_rate & (filtered_traces['value'] >= filtered_traces['p95'])
filtered_traces.loc[mask, 'high_anomaly'] = True
# if already has a kind, set to mixed later; for now label 'error'
filtered_traces.loc[mask & (filtered_traces['anomaly_kind'].isna()), 'anomaly_kind'] = 'error'
filtered_traces.loc[mask & (filtered_traces['anomaly_kind'].notna()) & (filtered_traces['anomaly_kind'] != 'error'), 'anomaly_kind'] = 'mixed'

# row_count -> low if value <= p5
mask = is_row_count & (filtered_traces['value'] <= filtered_traces['p5'])
filtered_traces.loc[mask, 'low_anomaly'] = True
filtered_traces.loc[mask & (filtered_traces['anomaly_kind'].isna()), 'anomaly_kind'] = 'row_count'
filtered_traces.loc[mask & (filtered_traces['anomaly_kind'].notna()) & (filtered_traces['anomaly_kind'] != 'row_count'), 'anomaly_kind'] = 'mixed'

# If a row ends up with both high and low (unlikely), mark mixed
both_mask = filtered_traces['high_anomaly'] & filtered_traces['low_anomaly']
filtered_traces.loc[both_mask, 'anomaly_kind'] = 'mixed'

# For rows with anomaly_kind still None but flagged, set to 'mixed'
filtered_traces.loc[(filtered_traces['anomaly_kind'].isna()) & ((filtered_traces['high_anomaly']) | (filtered_traces['low_anomaly'])), 'anomaly_kind'] = 'mixed'

# 4a) Total unique (cmdb_id, trace_name) pairs analyzed
unique_pairs_count = int(trace_threshold_df.shape[0])

# Prepare anomalies subset
trace_anoms = filtered_traces[(filtered_traces['high_anomaly']) | (filtered_traces['low_anomaly'])].copy()

# 4b) Number of pairs with >=1 duration/error high anomaly, and number with >=1 row_count low anomaly
# duration/error high:
dur_err_mask = trace_anoms['anomaly_kind'].isin(['duration','error','mixed'])
# But we must ensure 'duration' or 'error' type specifically: compute pairs where any high anomaly occurred for duration or error
high_pairs = trace_anoms[trace_anoms['high_anomaly']].groupby(['cmdb_id','trace_name']).size().reset_index(name='cnt')
# classify whether that trace_name is duration or error or other
def kind_of_trace(name):
    name = str(name)
    if 'duration' in name:
        return 'duration'
    if 'error_rate' in name:
        return 'error'
    if 'row_count' in name:
        return 'row_count'
    return 'other'
high_pairs['trace_kind'] = high_pairs['trace_name'].apply(kind_of_trace)
duration_error_high_pairs_count = int(high_pairs[high_pairs['trace_kind'].isin(['duration','error'])].shape[0])

# row_count low anomaly pairs
low_pairs = trace_anoms[trace_anoms['low_anomaly']].groupby(['cmdb_id','trace_name']).size().reset_index(name='cnt')
low_pairs['trace_kind'] = low_pairs['trace_name'].apply(kind_of_trace)
rowcount_low_pairs_count = int(low_pairs[low_pairs['trace_kind']=='row_count'].shape[0])

# 4c) Top 20 (cmdb_id, trace_name) pairs ranked by total anomaly count
if trace_anoms.empty:
    top20_trace_pairs_df = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','predominant_anomaly_kind'])
else:
    grp = trace_anoms.groupby(['cmdb_id','trace_name'], as_index=False)
    pair_stats = grp.agg(
        anomaly_count=('timestamp','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        duration_count=('anomaly_kind', lambda s: (s=='duration').sum()),
        error_count=('anomaly_kind', lambda s: (s=='error').sum()),
        rowcount_count=('anomaly_kind', lambda s: (s=='row_count').sum()),
        mixed_count=('anomaly_kind', lambda s: (s=='mixed').sum())
    )
    # determine predominant_anomaly_kind
    def predominant(r):
        counts = {'duration': r['duration_count'], 'error': r['error_count'], 'row_count': r['rowcount_count']}
        max_kind = max(counts, key=counts.get)
        # if tie or mixed heavy, label mixed
        if list(counts.values()).count(counts[max_kind]) > 1:
            return 'mixed'
        return max_kind
    pair_stats['predominant_anomaly_kind'] = pair_stats.apply(predominant, axis=1)
    pair_stats['earliest_anomaly_timestamp'] = pair_stats['earliest_anomaly_timestamp'].astype('Int64')
    top20_trace_pairs_df = pair_stats.sort_values(['anomaly_count','earliest_anomaly_timestamp'], ascending=[False,True])[
        ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','predominant_anomaly_kind']
    ].head(20).reset_index(drop=True)

# 4d) Top 10 destination components ranked by total trace anomalies
if trace_anoms.empty:
    top10_dest_components_df = pd.DataFrame(columns=['cmdb_id','total_trace_anomalies','earliest_anomaly_timestamp'])
else:
    comp_stats = trace_anoms.groupby('cmdb_id').agg(
        total_trace_anomalies=('timestamp','size'),
        earliest_anomaly_timestamp=('timestamp','min')
    ).reset_index()
    comp_stats['earliest_anomaly_timestamp'] = comp_stats['earliest_anomaly_timestamp'].astype('Int64')
    top10_dest_components_df = comp_stats.sort_values(['total_trace_anomalies','earliest_anomaly_timestamp'], ascending=[False,True]).head(10).reset_index(drop=True)

# 4e) Candidate set components report (only those present in filtered_traces)
candidates = [
    'adservice','adservice2','frontend-0','frontend-1','frontend-2','frontend2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
    'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2',
    'paymentservice-2','node-1','node-2','node-5'
]
cand_df = trace_anoms[trace_anoms['cmdb_id'].isin(candidates)].groupby('cmdb_id').agg(
    trace_anomaly_count=('timestamp','size'),
    earliest_anomaly_timestamp=('timestamp','min')
).reset_index()
if not cand_df.empty:
    cand_df['earliest_anomaly_timestamp'] = cand_df['earliest_anomaly_timestamp'].astype('Int64')

# Save trace_threshold_df and filtered_traces (with anomaly flags) in kernel variables (already assigned)

# Return compact aggregates only (as variables)
unique_pairs_count, duration_error_high_pairs_count, rowcount_low_pairs_count, top20_trace_pairs_df, top10_dest_components_df, cand_df

```
Out[4]:
```
```python
# Compact plain-English summary of trace-level anomaly detection results.
summary = (
    "Trace anomaly detection summary (incident window: 2022-03-20 23:30 to 2022-03-21 00:00 UTC+8):\n\n"
    "- Unique (cmdb_id, trace_name) pairs analyzed: 1056\n"
    "- Pairs with ≥1 duration/error HIGH anomaly in window: 792\n"
    "- Pairs with ≥1 row_count LOW anomaly in window: 264\n\n"
    "- Top example trace pairs by anomaly count (cmdb_id, trace_name, anomaly_count, earliest_ts, predominant_kind):\n"
    "  1) adservice-0, trace.from_frontend-0.error_rate, 30, 1647790200, error\n"
    "  2) adservice-0, trace.from_frontend-2.error_rate, 30, 1647790200, error\n"
    "  3) adservice-1, trace.from_frontend-0.error_rate, 30, 1647790200, error\n"
    "  4) adservice-1, trace.from_frontend-2.error_rate, 30, 1647790200, error\n"
    "  5) adservice-2, trace.from_frontend-0.error_rate, 30, 1647790200, error\n"
    "  (Many top pairs are service-targeted error_rate traces originating from frontend sources.)\n\n"
    "- Top 10 destination components by total trace anomalies (cmdb_id: total_trace_anomalies, earliest_anomaly_timestamp):\n"
    "  1) frontend-2: 859, 1647790200\n"
    "  2) frontend-0: 830, 1647790200\n"
    "  3) checkoutservice-1: 822, 1647790260\n"
    "  4) checkoutservice-0: 809, 1647790200\n"
    "  5) checkoutservice-2: 728, 1647790260\n"
    "  6) checkoutservice2-0: 409, 1647790200\n"
    "  7) productcatalogservice-2: 336, 1647790200\n"
    "  8) frontend2-0: 332, 1647790200\n"
    "  9) productcatalogservice-1: 332, 1647790200\n"
    " 10) productcatalogservice-0: 330, 1647790200\n\n"
    "- Candidate components (requested set) present in filtered traces (trace_anomaly_count, earliest_anomaly_timestamp):\n"
    "  - checkoutservice-0: 809, 1647790200\n"
    "  - checkoutservice-1: 822, 1647790260\n"
    "  - checkoutservice-2: 728, 1647790260\n"
    "  - checkoutservice2-0: 409, 1647790200\n"
    "  - frontend-0: 830, 1647790200\n"
    "  - frontend-2: 859, 1647790200\n"
    "  - frontend2-0: 332, 1647790200\n"
    "  - paymentservice-2: 63, 1647790260\n"
    "  - productcatalogservice-0: 330, 1647790200\n"
    "  - productcatalogservice-1: 332, 1647790200\n"
    "  - productcatalogservice-2: 336, 1647790200\n"
    "  (Other candidates were not present in filtered_traces or had zero trace anomalies.)\n\n"
    "Notes:\n"
    "- Timestamps are raw Unix seconds (UTC+8 context).\n"
    "- Many detected trace anomalies are error_rate spikes on traces from frontend -> backend (predominantly 'error').\n"
    "- trace_threshold_df and filtered_traces (with anomaly flags) are saved in kernel variables for the next log-correlation step."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(1056, 792, 264,               cmdb_id                         trace_name  anomaly_count  earliest_anomaly_timestamp predominant_anomaly_kind
0         adservice-0   trace.from_frontend-0.error_rate             30                  1647790200                    error
1         adservice-0   trace.from_frontend-2.error_rate             30                  1647790200                    error
2         adservice-1   trace.from_frontend-0.error_rate             30                  1647790200                    error
3         adservice-1   trace.from_frontend-2.error_rate             30                  1647790200                    error
4         adservice-2   trace.from_frontend-0.error_rate             30                  1647790200                    error
5         adservice-2   trace.from_frontend-2.error_rate             30                  1647790200                    error
6        adservice2-0  trace.from_frontend2-0.error_rate             30                  1647790200                    error
7       cartservice-0   trace.from_frontend-0.error_rate             30                  1647790200                    error
8       cartservice-0   trace.from_frontend-2.error_rate             30                  1647790200                    error
9       cartservice-0              trace.self.error_rate             30                  1647790200                    error
10      cartservice-1   trace.from_frontend-0.error_rate             30                  1647790200                    error
11      cartservice-1   trace.from_frontend-2.error_rate             30                  1647790200                    error
12      cartservice-1              trace.self.error_rate             30                  1647790200                    error
13      cartservice-2   trace.from_frontend-0.error_rate             30                  1647790200                    error
14      cartservice-2   trace.from_frontend-2.error_rate             30                  1647790200                    error
15      cartservice-2              trace.self.error_rate             30                  1647790200                    error
16     cartservice2-0  trace.from_frontend2-0.error_rate             30                  1647790200                    error
17     cartservice2-0              trace.self.error_rate             30                  1647790200                    error
18  currencyservice-0   trace.from_frontend-0.error_rate             30                  1647790200                    error
19  currencyservice-0   trace.from_frontend-2.error_rate             30                  1647790200                    error,                    cmdb_id  total_trace_anomalies  earliest_anomaly_timestamp
0               frontend-2                    859                  1647790200
1               frontend-0                    830                  1647790200
2        checkoutservice-1                    822                  1647790260
3        checkoutservice-0                    809                  1647790200
4        checkoutservice-2                    728                  1647790260
5       checkoutservice2-0                    409                  1647790200
6  productcatalogservice-2                    336                  1647790200
7              frontend2-0                    332                  1647790200
8  productcatalogservice-1                    332                  1647790200
9  productcatalogservice-0                    330                  1647790200,                     cmdb_id  trace_anomaly_count  earliest_anomaly_timestamp
0         checkoutservice-0                  809                  1647790200
1         checkoutservice-1                  822                  1647790260
2         checkoutservice-2                  728                  1647790260
3        checkoutservice2-0                  409                  1647790200
4                frontend-0                  830                  1647790200
5                frontend-2                  859                  1647790200
6               frontend2-0                  332                  1647790200
7          paymentservice-2                   63                  1647790260
8   productcatalogservice-0                  330                  1647790200
9   productcatalogservice-1                  332                  1647790200
10  productcatalogservice-2                  336                  1647790200)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: log_df, error_logs_df, start_ts, end_ts
tz_sh = pytz.timezone('Asia/Shanghai')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv('error_logs.csv')
try:
    start_ts, end_ts
except NameError:
    start_ts = int(pd.Timestamp('2022-03-20 23:30:00', tz=tz_sh).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-21 00:00:00', tz=tz_sh).timestamp())

# Ensure numeric types
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) Compute global thresholds (p95 and p5) per (cmdb_id, log_name)
log_threshold_df = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# 2) Filter log.csv to incident window
filtered_logs = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies
filtered_logs = filtered_logs.merge(log_threshold_df, on=['cmdb_id','log_name'], how='left')

# Initialize flags
filtered_logs['high_anomaly'] = False
filtered_logs['low_anomaly'] = False
filtered_logs['anomaly_kind'] = None

# high: log_name contains 'error' and value >= p95
mask_high = filtered_logs['log_name'].astype(str).str.contains('error', case=False, na=False) & (filtered_logs['value'] >= filtered_logs['p95'])
filtered_logs.loc[mask_high, 'high_anomaly'] = True
filtered_logs.loc[mask_high, 'anomaly_kind'] = 'error'

# low: log_name contains 'row_count' and value <= p5
mask_low = filtered_logs['log_name'].astype(str).str.contains('row_count', case=False, na=False) & (filtered_logs['value'] <= filtered_logs['p5'])
filtered_logs.loc[mask_low, 'low_anomaly'] = True
filtered_logs.loc[mask_low & (filtered_logs['anomaly_kind'].isna()), 'anomaly_kind'] = 'row_count'
filtered_logs.loc[mask_low & (filtered_logs['anomaly_kind'].notna()) & (filtered_logs['anomaly_kind'] != 'row_count'), 'anomaly_kind'] = 'mixed'

# For any rows with both flags, set kind to 'mixed'
both_mask = filtered_logs['high_anomaly'] & filtered_logs['low_anomaly']
filtered_logs.loc[both_mask, 'anomaly_kind'] = 'mixed'
filtered_logs.loc[(filtered_logs['anomaly_kind'].isna()) & ((filtered_logs['high_anomaly']) | (filtered_logs['low_anomaly'])), 'anomaly_kind'] = 'mixed'

# 4a) Count of unique (cmdb_id, log_name) pairs analyzed
unique_pairs_count = int(log_threshold_df.shape[0])

# Prepare anomalies subset
log_anoms = filtered_logs[(filtered_logs['high_anomaly']) | (filtered_logs['low_anomaly'])].copy()

# 4b) Number of pairs with >=1 high_anomaly and number with >=1 low_anomaly
high_pairs = log_anoms[log_anoms['high_anomaly']].groupby(['cmdb_id','log_name']).size().reset_index(name='cnt')
low_pairs  = log_anoms[log_anoms['low_anomaly']].groupby(['cmdb_id','log_name']).size().reset_index(name='cnt')
high_pairs_count = int(high_pairs.shape[0])
low_pairs_count = int(low_pairs.shape[0])

# 4c) Top 20 components ranked by total log anomalies
if log_anoms.empty:
    top20_components_df = pd.DataFrame(columns=['cmdb_id','total_log_anomalies','number_of_log_names_with_anomalies','earliest_anomaly_timestamp'])
else:
    comp_stats = log_anoms.groupby('cmdb_id').agg(
        total_log_anomalies=('timestamp','size'),
        number_of_log_names_with_anomalies=('log_name', pd.Series.nunique),
        earliest_anomaly_timestamp=('timestamp','min')
    ).reset_index()
    comp_stats['earliest_anomaly_timestamp'] = comp_stats['earliest_anomaly_timestamp'].astype('Int64')
    top20_components_df = comp_stats.sort_values(['total_log_anomalies','earliest_anomaly_timestamp'], ascending=[False,True]).head(20).reset_index(drop=True)

# 4d) Candidate components report
candidates = [
    'adservice','adservice2','frontend-0','frontend-1','frontend-2','frontend2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
    'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2',
    'paymentservice-2','node-1','node-2','node-5'
]
cand_subset = log_anoms[log_anoms['cmdb_id'].isin(candidates)].copy()
if cand_subset.empty:
    candidates_report_df = pd.DataFrame(columns=['cmdb_id','total_log_anomalies','earliest_anomaly_timestamp','log.error_count_high_anomaly'])
else:
    cand_stats = cand_subset.groupby('cmdb_id').agg(
        total_log_anomalies=('timestamp','size'),
        earliest_anomaly_timestamp=('timestamp','min')
    ).reset_index()
    cand_stats['earliest_anomaly_timestamp'] = cand_stats['earliest_anomaly_timestamp'].astype('Int64')
    # Whether log.error_count had high_anomaly for each component in the window
    # Look for rows in filtered_logs where log_name == 'log.error_count' and high_anomaly True
    err_high = filtered_logs[(filtered_logs['log_name'] == 'log.error_count') & (filtered_logs['high_anomaly'])]
    err_high_flag = err_high.groupby('cmdb_id').size().reset_index(name='err_high_count')
    cand_stats = cand_stats.merge(err_high_flag, on='cmdb_id', how='left')
    cand_stats['log.error_count_high_anomaly'] = cand_stats['err_high_count'].fillna(0).astype(int) > 0
    candidates_report_df = cand_stats[['cmdb_id','total_log_anomalies','earliest_anomaly_timestamp','log.error_count_high_anomaly']].copy()

# 4e) Check error_logs.csv for rows in the incident window
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce').astype('Int64')
err_window = error_logs_df[(error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] <= end_ts)].copy()
if err_window.empty:
    # Create a small DataFrame with a message
    error_window_samples = pd.DataFrame([{'note': 'error_logs.csv has no rows in the incident window'}])
else:
    # return up to 5 sample messages (cmdb_id, timestamp, message)
    error_window_samples = err_window[['cmdb_id','timestamp','message']].head(5).copy()
    error_window_samples['timestamp'] = error_window_samples['timestamp'].astype('Int64')

# Save log_threshold_df and filtered_logs in kernel vars (already assigned)

# Return only requested compact aggregates
unique_pairs_count, high_pairs_count, low_pairs_count, top20_components_df, candidates_report_df, error_window_samples

```
Out[5]:
```
```python
# Compact plain-English summary of log-level anomaly detection results (incident window: 2022-03-20 23:30 to 2022-03-21 00:00 UTC+8).
summary = (
    "Log anomaly detection summary:\n\n"
    "- Unique (cmdb_id, log_name) pairs analyzed: 62\n"
    "- Pairs with ≥1 high_anomaly: 31\n"
    "- Pairs with ≥1 low_anomaly: 31\n\n"
    "- Top 20 components by total log anomalies (cmdb_id: total_log_anomalies, number_of_log_names_with_anomalies, earliest_anomaly_timestamp):\n"
    "  1) recommendationservice-1: 34, 2, 1647790200\n"
    "  2) cartservice-1: 33, 2, 1647790200\n"
    "  3) cartservice-2: 33, 2, 1647790200\n"
    "  4) currencyservice-0: 33, 2, 1647790200\n"
    "  5) currencyservice-1: 33, 2, 1647790200\n"
    "  6) currencyservice-2: 33, 2, 1647790200\n"
    "  7) frontend-0: 33, 2, 1647790200\n"
    "  8) shippingservice-1: 33, 2, 1647790200\n"
    "  9) shippingservice-2: 33, 2, 1647790200\n"
    " 10) adservice-0: 32, 2, 1647790200\n"
    " 11) adservice-1: 32, 2, 1647790200\n"
    " 12) adservice-2: 32, 2, 1647790200\n"
    " 13) cartservice-0: 32, 2, 1647790200\n"
    " 14) frontend-2: 32, 2, 1647790200\n"
    " 15) productcatalogservice-0: 32, 2, 1647790200\n"
    " 16) productcatalogservice-1: 32, 2, 1647790200\n"
    " 17) productcatalogservice-2: 32, 2, 1647790200\n"
    " 18) recommendationservice-0: 32, 2, 1647790200\n"
    " 19) recommendationservice-2: 32, 2, 1647790200\n"
    " 20) shippingservice-0: 32, 2, 1647790200\n\n"
    "- Candidate components report (only those present in filtered logs):\n"
    "  - checkoutservice-0: total_log_anomalies=30, earliest=1647790200, log.error_count_high_anomaly=True\n"
    "  - checkoutservice-1: total_log_anomalies=28, earliest=1647790260, log.error_count_high_anomaly=True\n"
    "  - checkoutservice-2: total_log_anomalies=27, earliest=1647790260, log.error_count_high_anomaly=True\n"
    "  - frontend-0: total_log_anomalies=33, earliest=1647790200, log.error_count_high_anomaly=True\n"
    "  - frontend-1: total_log_anomalies=2,  earliest=1647790620, log.error_count_high_anomaly=True\n"
    "  - frontend-2: total_log_anomalies=32, earliest=1647790200, log.error_count_high_anomaly=True\n"
    "  - paymentservice-2: total_log_anomalies=23, earliest=1647790260, log.error_count_high_anomaly=True\n"
    "  - productcatalogservice-0: total_log_anomalies=32, earliest=1647790200, log.error_count_high_anomaly=True\n"
    "  - productcatalogservice-1: total_log_anomalies=32, earliest=1647790200, log.error_count_high_anomaly=True\n"
    "  - productcatalogservice-2: total_log_anomalies=32, earliest=1647790200, log.error_count_high_anomaly=True\n\n"
    "- error_logs.csv in the incident window: no rows (empty for this window).\n\n"
    "Notes:\n"
    "- Timestamps are raw Unix seconds (UTC+8 context).\n"
    "- log_threshold_df and filtered_logs (with anomaly flags) are saved in kernel variables for follow-up correlation."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(62, 31, 31,                     cmdb_id  total_log_anomalies  number_of_log_names_with_anomalies  earliest_anomaly_timestamp
0   recommendationservice-1                   34                                   2                  1647790200
1             cartservice-1                   33                                   2                  1647790200
2             cartservice-2                   33                                   2                  1647790200
3         currencyservice-0                   33                                   2                  1647790200
4         currencyservice-1                   33                                   2                  1647790200
5         currencyservice-2                   33                                   2                  1647790200
6                frontend-0                   33                                   2                  1647790200
7         shippingservice-1                   33                                   2                  1647790200
8         shippingservice-2                   33                                   2                  1647790200
9               adservice-0                   32                                   2                  1647790200
10              adservice-1                   32                                   2                  1647790200
11              adservice-2                   32                                   2                  1647790200
12            cartservice-0                   32                                   2                  1647790200
13               frontend-2                   32                                   2                  1647790200
14  productcatalogservice-0                   32                                   2                  1647790200
15  productcatalogservice-1                   32                                   2                  1647790200
16  productcatalogservice-2                   32                                   2                  1647790200
17  recommendationservice-0                   32                                   2                  1647790200
18  recommendationservice-2                   32                                   2                  1647790200
19        shippingservice-0                   32                                   2                  1647790200,                    cmdb_id  total_log_anomalies  earliest_anomaly_timestamp  log.error_count_high_anomaly
0        checkoutservice-0                   30                  1647790200                          True
1        checkoutservice-1                   28                  1647790260                          True
2        checkoutservice-2                   27                  1647790260                          True
3               frontend-0                   33                  1647790200                          True
4               frontend-1                    2                  1647790620                          True
5               frontend-2                   32                  1647790200                          True
6         paymentservice-2                   23                  1647790260                          True
7  productcatalogservice-0                   32                  1647790200                          True
8  productcatalogservice-1                   32                  1647790200                          True
9  productcatalogservice-2                   32                  1647790200                          True,                                                 note
0  error_logs.csv has no rows in the incident window)```
```

In [ ]:
# In[6]:

import pandas as pd
import re
import pytz

# Reuse kernel variables: faults_df, filtered_traces, filtered_logs
# Define normalization function
def normalize_cmdb(cm):
    if pd.isna(cm):
        return cm
    cm = str(cm)
    # keep node names as-is
    if re.match(r'^node-\d+$', cm):
        return cm
    # remove trailing -number suffix for pods (e.g., adservice-0 -> adservice)
    m = re.match(r'^(.+?)-\d+$', cm)
    if m:
        return m.group(1)
    return cm

# Add service columns (normalized) to the three dataframes (create copies to avoid altering original structures unexpectedly)
faults_df['service'] = faults_df['cmdb_id'].apply(normalize_cmdb)
filtered_traces['service'] = filtered_traces['cmdb_id'].apply(normalize_cmdb)
filtered_logs['service'] = filtered_logs['cmdb_id'].apply(normalize_cmdb)

# 2) Compute aggregates per service

# pods_with_faults: distinct pod cmdb_ids with >=1 fault run
pods_with_faults_s = faults_df.groupby('service')['cmdb_id'].nunique().rename('pods_with_faults')

# total_fault_runs: total count of runs across pods (rows in faults_df)
total_fault_runs_s = faults_df.groupby('service').size().rename('total_fault_runs')

# earliest_fault_ts: min run_start_ts per service
earliest_fault_ts_s = faults_df.groupby('service')['run_start_ts'].min().rename('earliest_fault_ts')

# trace_anomalies: total number of anomalous trace rows in filtered_traces where destination maps to this service
trace_anoms = filtered_traces[(filtered_traces['high_anomaly'] == True) | (filtered_traces['low_anomaly'] == True)].copy()
trace_anomalies_s = trace_anoms.groupby('service').size().rename('trace_anomalies')

# log_anomalies: total number of anomalous log rows in filtered_logs for pods of this service
log_anoms = filtered_logs[(filtered_logs['high_anomaly'] == True) | (filtered_logs['low_anomaly'] == True)].copy()
log_anomalies_s = log_anoms.groupby('service').size().rename('log_anomalies')

# all_pods_log_error_high:
# For each pod in filtered_logs, determine if it has any log.error_count high anomaly
err_high_per_pod = (
    filtered_logs[
        (filtered_logs['log_name'] == 'log.error_count') & (filtered_logs['high_anomaly'] == True)
    ]
    .groupby('cmdb_id').size().rename('err_high_count')
    .reset_index()
)
# pods present in filtered_logs
pods_in_filtered_logs = filtered_logs[['cmdb_id','service']].drop_duplicates()

# Merge presence and err_high_count
pods_err = pods_in_filtered_logs.merge(err_high_per_pod, on='cmdb_id', how='left')
pods_err['err_high_flag'] = pods_err['err_high_count'].fillna(0).astype(int) > 0

# Now per service, check if every pod present in filtered_logs has err_high_flag == True
service_err_group = pods_err.groupby('service').agg(
    number_of_pods_in_filtered_logs = ('cmdb_id','nunique'),
    pods_with_err_high = ('err_high_flag','sum')
).reset_index()
service_err_group['all_pods_log_error_high'] = service_err_group['pods_with_err_high'] >= service_err_group['number_of_pods_in_filtered_logs']
service_err_group = service_err_group.set_index('service')['all_pods_log_error_high']

# Union of services across datasets
all_services = set(pods_with_faults_s.index) | set(trace_anomalies_s.index) | set(log_anomalies_s.index) | set(service_err_group.index)

# Build summary DataFrame
service_summary = pd.DataFrame(sorted(list(all_services)), columns=['service']).set_index('service')

service_summary = service_summary.join(pods_with_faults_s).join(total_fault_runs_s).join(earliest_fault_ts_s)
service_summary = service_summary.join(trace_anomalies_s).join(log_anomalies_s).join(service_err_group)

# Fill NaNs: numeric zeros where appropriate, earliest_fault_ts keep as NA if missing
service_summary['pods_with_faults'] = service_summary['pods_with_faults'].fillna(0).astype(int)
service_summary['total_fault_runs'] = service_summary['total_fault_runs'].fillna(0).astype(int)
service_summary['trace_anomalies'] = service_summary['trace_anomalies'].fillna(0).astype(int)
service_summary['log_anomalies'] = service_summary['log_anomalies'].fillna(0).astype(int)
service_summary['all_pods_log_error_high'] = service_summary['all_pods_log_error_high'].fillna(False).astype(bool)
# earliest_fault_ts: keep Int64 with NA where missing
service_summary['earliest_fault_ts'] = service_summary['earliest_fault_ts'].astype('Int64')

# Reset index for display and sort by total_fault_runs descending, limit to top 20
service_summary = service_summary.reset_index().sort_values('total_fault_runs', ascending=False).head(20).reset_index(drop=True)

# Keep only requested columns in the specified order
service_summary = service_summary[['service','pods_with_faults','total_fault_runs','earliest_fault_ts','trace_anomalies','log_anomalies','all_pods_log_error_high']]

# Save service_summary in kernel variable and display compact result (top 20)
service_summary

```
Out[6]:
```
```python
# Produce a plain-English summary of the service-level aggregates stored in service_summary.
rows = service_summary.to_dict('records')

top5 = rows[:5]
top5_lines = []
for r in top5:
    top5_lines.append(f"{r['service']}: total_fault_runs={r['total_fault_runs']}, pods_with_faults={r['pods_with_faults']}, earliest_fault_ts={int(r['earliest_fault_ts'])}, trace_anomalies={r['trace_anomalies']}, log_anomalies={r['log_anomalies']}, all_pods_log_error_high={bool(r['all_pods_log_error_high'])}")

# Services where every pod present in logs had log.error_count high anomaly
all_error_services = [r['service'] for r in rows if r['all_pods_log_error_high']]
some_no_error_services = [r['service'] for r in rows if not r['all_pods_log_error_high']]

summary = (
    "Service-level fault summary (compact):\n\n"
    f"Top services by total fault runs (showing top 5):\n  - " + "\n  - ".join(top5_lines) + "\n\n"
    f"Highlights:\n"
    f"- Service with the most fault runs: {rows[0]['service']} ({rows[0]['total_fault_runs']} runs).\n"
    f"- Several core services show both many fault runs and many trace anomalies (checkoutservice, frontend, adservice, productcatalogservice, cartservice).\n"
    f"- Services where every pod present in logs had log.error_count high anomaly: {all_error_services}\n"
    f"- Services where not all pods had log.error_count high anomaly: {some_no_error_services}\n\n"
    "Note: timestamps are raw Unix seconds (UTC+8 context). 'service_summary' is saved in the kernel for follow-up."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

service  pods_with_faults  total_fault_runs  earliest_fault_ts  trace_anomalies  log_anomalies  all_pods_log_error_high
0          checkoutservice                 4              1336         1647790200             2359             85                     True
1                 frontend                 4              1018         1647790200             1689             67                     True
2                adservice                 4               869         1647790200              223             96                     True
3    productcatalogservice                 4               732         1647790200              998             96                     True
4              cartservice                 4               565         1647790200              834             98                     True
5               adservice2                 2               544         1647790200               39              0                    False
6          shippingservice                 4               519         1647790200              547             98                     True
7    recommendationservice                 4               518         1647790200              672             98                     True
8          currencyservice                 4               515         1647790200              596             99                     True
9             emailservice                 4               503         1647790200              201             81                     True
10          paymentservice                 4               426         1647790200              204             75                     True
11        checkoutservice2                 1               360         1647790200              409              0                    False
12               frontend2                 1               343         1647790200              332              0                    False
13                  node-4                 1               196         1647790200                0              0                    False
14  productcatalogservice2                 1               190         1647790200               80              0                    False
15                  node-3                 1               181         1647790200                0              0                    False
16        shippingservice2                 1               179         1647790200               77              0                    False
17            cartservice2                 1               178         1647790200              130              0                    False
18                  node-2                 1               164         1647790200                0              0                    False
19  recommendationservice2                 1               163         1647790200              186              0                    False```
```

In [ ]:
# In[7]:

import pandas as pd
import re
import pytz

# Reuse kernel variables: filtered_traces, faults_df, service_summary
# Candidate services
candidates = [
    'checkoutservice','frontend','adservice','productcatalogservice',
    'cartservice','paymentservice','node-1','node-2','node-5'
]

# Normalization function (reuse logic from prior step)
def normalize_cmdb(cm):
    if pd.isna(cm):
        return cm
    cm = str(cm)
    if re.match(r'^node-\d+$', cm):
        return cm
    m = re.match(r'^(.+?)-\d+$', cm)
    if m:
        return m.group(1)
    return cm

# Work on anomalous filtered_traces (rows already have high_anomaly/low_anomaly flags)
df_anoms = filtered_traces[(filtered_traces['high_anomaly'] == True) | (filtered_traces['low_anomaly'] == True)].copy()

# Extract trace_origin when trace_name starts with 'trace.from_<origin>.'
# pattern: trace.from_<origin>.<rest>
df_anoms['trace_origin'] = df_anoms['trace_name'].astype(str).str.extract(r'^trace\.from_([^\.]+)\.', expand=False)

# Normalize destination and origin to service names
df_anoms['dest_service'] = df_anoms['cmdb_id'].apply(normalize_cmdb)
df_anoms['origin_service'] = df_anoms['trace_origin'].apply(normalize_cmdb)

# Compute anomalous incoming traces per destination service
incoming = df_anoms.groupby('dest_service').size().rename('anomalous_incoming_traces').reset_index()
# Compute anomalous outgoing traces per origin service (only where origin parsed)
outgoing = df_anoms[~df_anoms['origin_service'].isna()].groupby('origin_service').size().rename('anomalous_outgoing_traces').reset_index()

# Earliest trace anomaly timestamp per service (considering both incoming and outgoing)
# For each service, consider rows where dest_service==service OR origin_service==service
# Build a long mapping of service -> timestamps
incoming_ts = df_anoms[['dest_service','timestamp']].rename(columns={'dest_service':'service'})
outgoing_ts = df_anoms[~df_anoms['origin_service'].isna()][['origin_service','timestamp']].rename(columns={'origin_service':'service'})
all_ts = pd.concat([incoming_ts, outgoing_ts], ignore_index=True)
earliest_ts = all_ts.groupby('service')['timestamp'].min().rename('earliest_trace_anomaly_ts').reset_index()

# Build base result for candidate services
svc_df = pd.DataFrame({'service': candidates})

# Merge counts and earliest_ts (may produce NaN for services with no entries)
svc_df = svc_df.merge(incoming.rename(columns={'dest_service':'service'}), on='service', how='left')
svc_df = svc_df.merge(outgoing.rename(columns={'origin_service':'service'}), on='service', how='left')
svc_df = svc_df.merge(earliest_ts, on='service', how='left')

# Fill NaNs with zeros/NA as required
svc_df['anomalous_incoming_traces'] = svc_df['anomalous_incoming_traces'].fillna(0).astype(int)
svc_df['anomalous_outgoing_traces'] = svc_df['anomalous_outgoing_traces'].fillna(0).astype(int)
svc_df['earliest_trace_anomaly_ts'] = svc_df['earliest_trace_anomaly_ts'].astype('Int64')

# Merge pods_with_faults and earliest_metric_fault_ts and log_error_high from service_summary if present
ss = service_summary.set_index('service')
svc_df['pods_with_faults'] = svc_df['service'].map(lambda s: int(ss.loc[s,'pods_with_faults']) if s in ss.index else 0)
# earliest_metric_fault_ts: use earliest_fault_ts from service_summary
def get_earliest_metric_ts(s):
    if s in ss.index:
        v = ss.loc[s,'earliest_fault_ts']
        return int(v) if pd.notna(v) else pd.NA
    return pd.NA
svc_df['earliest_metric_fault_ts'] = svc_df['service'].apply(get_earliest_metric_ts).astype('Int64')
# log_error_high flag
svc_df['log_error_high'] = svc_df['service'].map(lambda s: bool(ss.loc[s,'all_pods_log_error_high']) if s in ss.index else False)

# Final selection and sort by anomalous_incoming_traces descending, limit to top 20
service_downstream_df = svc_df.sort_values('anomalous_incoming_traces', ascending=False).head(20).reset_index(drop=True)[
    ['service','anomalous_incoming_traces','anomalous_outgoing_traces','earliest_trace_anomaly_ts','pods_with_faults','earliest_metric_fault_ts','log_error_high']
]

service_downstream_df

```
Out[7]:
```
```python
# Summarize the downstream-analysis results in plain English (compact).
from datetime import datetime
import pytz

# Use service_downstream_df from kernel
tz = pytz.timezone('Asia/Shanghai')

def ts_to_str(ts):
    try:
        if ts is None or (pd.isna(ts)):
            return "N/A"
        return datetime.fromtimestamp(int(ts), tz).strftime("%Y-03-20 %H:%M:%S") + f" (raw:{int(ts)})"
    except Exception:
        return f"raw:{ts}"

import pandas as pd
rows = service_downstream_df.to_dict('records')

lines = []
lines.append("Downstream analysis summary (incident window 2022-03-20 23:30–00:00 UTC+8):")
lines.append("Key candidate services (service: incoming, outgoing, earliest_trace_anom, pods_with_faults, earliest_metric_fault_ts, all_pods_log_error_high):")
for r in rows:
    et = r['earliest_trace_anomaly_ts']
    em = r['earliest_metric_fault_ts']
    et_str = ts_to_str(et) if et is not pd.NA else "N/A"
    em_str = ts_to_str(em) if em is not pd.NA else "N/A"
    lines.append(
        f"- {r['service']}: incoming={r['anomalous_incoming_traces']}, outgoing={r['anomalous_outgoing_traces']}, "
        f"earliest_trace_anom={et_str}, pods_with_faults={r['pods_with_faults']}, earliest_metric_fault_ts={em_str}, "
        f"log_error_high={bool(r['log_error_high'])}"
    )

lines.append("\nInterpretation:")
lines.append("- checkoutservice and frontend show the largest number of anomalous incoming traces (checkoutservice highest), and both also have large outgoing anomaly counts, indicating heavy involvement in the anomaly graph.")
lines.append("- productcatalogservice, cartservice, adservice, and paymentservice show substantial incoming anomalies but zero outgoing anomalies in traces (they are more downstream targets), with earliest trace anomalies and metric faults at the same early timestamp (raw 1647790200).")
lines.append("- All listed core services have pods_with_faults >0 and their pods show log.error_count high anomalies (log_error_high=True), strengthening their candidacy for root-cause investigation.")
lines.append("- node-1, node-2, node-5 show no anomalous incoming traces in this filtered set (nodes not primary downstream candidates here).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

service  anomalous_incoming_traces  anomalous_outgoing_traces  earliest_trace_anomaly_ts  pods_with_faults  earliest_metric_fault_ts  log_error_high
0        checkoutservice                       2359                       1921                 1647790200                 4                1647790200            True
1               frontend                       1689                       1542                 1647790200                 4                1647790200            True
2  productcatalogservice                        998                          0                 1647790200                 4                1647790200            True
3            cartservice                        834                          0                 1647790200                 4                1647790200            True
4              adservice                        223                          0                 1647790200                 4                1647790200            True
5         paymentservice                        204                          0                 1647790200                 4                1647790200            True
6                 node-1                          0                          0                       <NA>                 0                      <NA>           False
7                 node-2                          0                          0                       <NA>                 1                1647790200           False
8                 node-5                          0                          0                       <NA>                 0                      <NA>           False```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse faults_df from kernel
# 1) Filter faults for productcatalogservice pods (including productcatalogservice2-0)
mask_prod = faults_df['cmdb_id'].astype(str).str.startswith('productcatalogservice')
productcatalog_faults = faults_df[mask_prod].copy()

# Ensure numeric formatting: round max_relative_severity for compact display
productcatalog_faults['max_relative_severity'] = pd.to_numeric(productcatalog_faults['max_relative_severity'], errors='coerce')

# 1) Top 20 fault runs for these pods sorted by max_relative_severity
top20_runs_df = productcatalog_faults.sort_values('max_relative_severity', ascending=False)[
    ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity']
].head(20).copy()
top20_runs_df['max_relative_severity'] = top20_runs_df['max_relative_severity'].round(6)
top20_runs_df['run_start_ts'] = top20_runs_df['run_start_ts'].astype('Int64')
top20_runs_df['run_end_ts'] = top20_runs_df['run_end_ts'].astype('Int64')

# 2) Counts of fault runs grouped by KPI category prefix (substring before first dot)
productcatalog_faults['kpi_category'] = productcatalog_faults['kpi_name'].astype(str).str.split('.', n=1).str[0]
category_counts = (
    productcatalog_faults
    .groupby('kpi_category', as_index=False)
    .agg(run_count=('kpi_name','size'), earliest_run_start_ts=('run_start_ts','min'))
    .sort_values(['run_count','earliest_run_start_ts'], ascending=[False,True])
)
category_counts['earliest_run_start_ts'] = category_counts['earliest_run_start_ts'].astype('Int64')
category_counts_df = category_counts.head(20).copy()

# 3) Container-related KPIs: filter by specified substrings
ctr_mask = productcatalog_faults['kpi_name'].astype(str).str.contains(r'container|system\.io|container\.fs|container_network', case=False, na=False)
container_related = productcatalog_faults[ctr_mask].copy()

container_kpis = (
    container_related
    .groupby('kpi_name', as_index=False)
    .agg(run_count=('kpi_name','size'),
         earliest_run_start_ts=('run_start_ts','min'),
         max_relative_severity=('max_relative_severity','max'))
    .sort_values(['run_count','max_relative_severity'], ascending=[False,False])
)
container_kpis['earliest_run_start_ts'] = container_kpis['earliest_run_start_ts'].astype('Int64')
container_kpis['max_relative_severity'] = container_kpis['max_relative_severity'].round(6)
container_kpis_df = container_kpis.head(10).copy()

# 4) Earliest run_start_ts overall and for container.* and system.io.* separately
if productcatalog_faults.empty:
    earliest_overall = pd.Series({'earliest_run_start_ts': pd.NA})
else:
    earliest_overall_ts = int(productcatalog_faults['run_start_ts'].min())
    earliest_overall = pd.Series({'earliest_run_start_ts': earliest_overall_ts})

# container-related earliest
if container_related.empty:
    earliest_container = pd.Series({'earliest_container_run_start_ts': pd.NA})
else:
    earliest_container_ts = int(container_related['run_start_ts'].min())
    earliest_container = pd.Series({'earliest_container_run_start_ts': earliest_container_ts})

# system.io.* earliest (explicit match)
sysio_mask = productcatalog_faults['kpi_name'].astype(str).str.contains(r'^system\.io', case=False, na=False)
sysio_related = productcatalog_faults[sysio_mask]
if sysio_related.empty:
    earliest_system_io = pd.Series({'earliest_system_io_run_start_ts': pd.NA})
else:
    earliest_system_io_ts = int(sysio_related['run_start_ts'].min())
    earliest_system_io = pd.Series({'earliest_system_io_run_start_ts': earliest_system_io_ts})

# Combine earliest summaries into a small DataFrame for compact display
earliest_summary_df = pd.DataFrame([{
    'earliest_overall_run_start_ts': earliest_overall['earliest_run_start_ts'],
    'earliest_container_run_start_ts': earliest_container['earliest_container_run_start_ts'],
    'earliest_system_io_run_start_ts': earliest_system_io['earliest_system_io_run_start_ts']
}])

# Final compact outputs (limited rows as requested)
top20_runs_df, category_counts_df, container_kpis_df, earliest_summary_df, productcatalog_faults.shape

```
Out[8]:
```
Summary of productcatalogservice fault-run analysis (compact):

- Total productcatalogservice fault runs found: 922 runs (productcatalog_faults saved in kernel).

1) Top fault runs by max_relative_severity (top 5 examples):
  - productcatalogservice-0, container.node-6.container_fs_reads./dev/vda, 1647790200–1647792000, 31 min, mixed, max_relative_severity=632.0
  - productcatalogservice-0, container.node-6.container_fs_reads_MB./dev/vda, 1647790200–1647792000, 31 min, mixed, max_relative_severity=135.608696
  - productcatalogservice2-0, container.node-6.container_memory_failures... (hierarchy), 1647790200–1647792000, 31 min, mixed, max_relative_severity=10.0
  - productcatalogservice2-0, container.node-6.container_memory_failures... (count), 1647790200–1647792000, 31 min, mixed, max_relative_severity=10.0
  - productcatalogservice-2, container.node-6.container_fs_reads_MB./dev/vda, 1647790200–1647790800, 11 min, mixed, max_relative_severity≈3.335607

2) Fault-run counts by KPI category prefix (top categories):
  - mesh: 500 runs, earliest_run_start_ts = 1647790200
  - container: 416 runs, earliest_run_start_ts = 1647790200
  - app: 6 runs, earliest_run_start_ts = 1647790200

3) Top container-related KPIs (top 10 by run_count) with earliest run and max severity:
  - container.node-6.container_cpu_user_seconds — run_count=19, earliest=1647790260, max_rel_severity=0.271109
  - container.node-6.container_cpu_cfs_periods — run_count=19, earliest=1647790260, max_rel_severity=0.155411
  - container.node-6.container_cpu_usage_seconds — run_count=18, earliest=1647790260, max_rel_severity=0.171142
  - container.node-6.container_cpu_cfs_throttled_p... — run_count=17, earliest=1647790260, max_rel_severity=0.173554
  - container.node-6.container_cpu_cfs_throttled_s... — run_count=15, earliest=1647790380, max_rel_severity=1.0
  - container.node-6.container_network_receive_MB.... — run_count=15, earliest=1647790380, max_rel_severity≈1.034696
  - container.node-6.container_network_transmit_MB... — run_count=14, earliest=1647790380, max_rel_severity≈0.269596
  - container.node-6.container_file_descriptors — run_count=13, earliest=1647790260, max_rel_severity≈0.268293
  - container.node-6.container_memory_failures.count/... — run_count=13, earliest=1647790200, max_rel_severity≈2.376289
  - container.node-6.container_memory_failures.hierarchy/... — run_count=13, earliest=1647790200, max_rel_severity≈2.376289

4) Earliest run_start timestamps:
  - Earliest overall productcatalogservice fault run: 1647790200 (2022-03-20 23:30:00 UTC+8)
  - Earliest container-related fault run: 1647790200 (2022-03-20 23:30:00 UTC+8)
  - Earliest system.io.* run: none found (NA)

Notes:
- Many of the highest-severity runs are container filesystem reads on node-6 (very large relative severity for fs_reads./dev/vda), and several container memory failure metrics show elevated severities.
- Mesh-category runs are numerous (500) indicating heavy inter-service trace-related anomalies involving productcatalogservice.
- productcatalog_faults is stored in the kernel for follow-up investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                           kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity
8113   productcatalogservice-0       container.node-6.container_fs_reads./dev/vda    1647790200  1647792000                  31                mixed             632.000000
8115   productcatalogservice-0    container.node-6.container_fs_reads_MB./dev/vda    1647790200  1647792000                  31                mixed             135.608696
8872  productcatalogservice2-0  container.node-6.container_memory_failures.hie...    1647790200  1647792000                  31                mixed              10.000000
8867  productcatalogservice2-0  container.node-6.container_memory_failures.con...    1647790200  1647792000                  31                mixed              10.000000
8583   productcatalogservice-2    container.node-6.container_fs_reads_MB./dev/vda    1647790200  1647790800                  11                mixed               3.335607
8926  productcatalogservice2-0  mesh.destination.checkoutservice2.productcatal...    1647790620  1647790800                   4                mixed               2.756511
8850  productcatalogservice2-0    container.node-6.container_fs_reads_MB./dev/vda    1647790740  1647792000                  22                mixed               2.378799
8375   productcatalogservice-1  container.node-6.container_memory_failures.con...    1647790200  1647790200                   1                 high               2.376289
8379   productcatalogservice-1  container.node-6.container_memory_failures.hie...    1647790200  1647790200                   1                 high               2.376289
8381   productcatalogservice-1  container.node-6.container_memory_failures.hie...    1647790680  1647792000                  23                mixed               2.206186
8377   productcatalogservice-1  container.node-6.container_memory_failures.con...    1647790680  1647792000                  23                mixed               2.206186
8128   productcatalogservice-0  container.node-6.container_memory_failures.con...    1647790200  1647790200                   1                 high               2.076190
8132   productcatalogservice-0  container.node-6.container_memory_failures.hie...    1647790200  1647790200                   1                 high               2.076190
8656   productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...    1647790440  1647790680                   5                mixed               1.903846
8558   productcatalogservice-2      container.node-6.container_cpu_system_seconds    1647790200  1647790320                   3                 high               1.872727
8608   productcatalogservice-2              container.node-6.container_memory_rss    1647790260  1647790680                   8                 high               1.839728
8876  productcatalogservice2-0              container.node-6.container_memory_rss    1647790560  1647791220                  12                mixed               1.743727
8580   productcatalogservice-2       container.node-6.container_fs_reads./dev/vda    1647790200  1647790800                  11                mixed               1.661355
8602   productcatalogservice-2  container.node-6.container_memory_failures.hie...    1647790680  1647790740                   2                 high               1.627907
8598   productcatalogservice-2  container.node-6.container_memory_failures.con...    1647790680  1647790740                   2                 high               1.627907,   kpi_category  run_count  earliest_run_start_ts
2         mesh        500             1647790200
1    container        416             1647790200
0          app          6             1647790200,                                              kpi_name  run_count  earliest_run_start_ts  max_relative_severity
6         container.node-6.container_cpu_user_seconds         19             1647790260               0.271109
0          container.node-6.container_cpu_cfs_periods         19             1647790260               0.155411
5        container.node-6.container_cpu_usage_seconds         18             1647790260               0.171142
1   container.node-6.container_cpu_cfs_throttled_p...         17             1647790260               0.173554
40  container.node-6.container_network_receive_MB....         15             1647790380               1.034696
2   container.node-6.container_cpu_cfs_throttled_s...         15             1647790380               1.000000
44  container.node-6.container_network_transmit_MB...         14             1647790380               0.269596
30  container.node-6.container_memory_failures.con...         13             1647790200               2.376289
32  container.node-6.container_memory_failures.hie...         13             1647790200               2.376289
7         container.node-6.container_file_descriptors         13             1647790260               0.268293,    earliest_overall_run_start_ts  earliest_container_run_start_ts earliest_system_io_run_start_ts
0                     1647790200                       1647790200                            <NA>, (922, 12))```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse faults_df and productcatalog_faults from kernel

# Ensure numeric severity
faults_df['max_relative_severity'] = pd.to_numeric(faults_df['max_relative_severity'], errors='coerce')

# 1) Select rows where kpi_name contains 'node-6' (case-sensitive)
node6_kpi_faults = faults_df[faults_df['kpi_name'].astype(str).str.contains('node-6', case=True, na=False)].copy()
node6_kpi_faults = node6_kpi_faults.sort_values('max_relative_severity', ascending=False, na_position='last')
# select and format columns for compact display
node6_kpi_faults_display = node6_kpi_faults[
    ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity']
].head(20).copy()
node6_kpi_faults_display['max_relative_severity'] = node6_kpi_faults_display['max_relative_severity'].round(6)
node6_kpi_faults_display['run_start_ts'] = node6_kpi_faults_display['run_start_ts'].astype('Int64')
node6_kpi_faults_display['run_end_ts'] = node6_kpi_faults_display['run_end_ts'].astype('Int64')

# 2) Total number and earliest run_start_ts among kpi_name containing 'node-6'
node6_kpi_count = int(node6_kpi_faults.shape[0])
if node6_kpi_count > 0:
    node6_kpi_earliest = int(node6_kpi_faults['run_start_ts'].min())
else:
    node6_kpi_earliest = pd.NA

# 3) Rows where cmdb_id == 'node-6' (node-level faults)
node6_cmdb_faults = faults_df[faults_df['cmdb_id'] == 'node-6'].copy()
node6_cmdb_faults = node6_cmdb_faults.sort_values('max_relative_severity', ascending=False, na_position='last')
node6_cmdb_faults_display = node6_cmdb_faults[
    ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity']
].head(10).copy()
node6_cmdb_faults_display['max_relative_severity'] = node6_cmdb_faults_display['max_relative_severity'].round(6)
node6_cmdb_faults_display['run_start_ts'] = node6_cmdb_faults_display['run_start_ts'].astype('Int64')
node6_cmdb_faults_display['run_end_ts'] = node6_cmdb_faults_display['run_end_ts'].astype('Int64')

# Count and earliest for node-level faults
node6_cmdb_count = int(node6_cmdb_faults.shape[0])
if node6_cmdb_count > 0:
    node6_cmdb_earliest = int(node6_cmdb_faults['run_start_ts'].min())
else:
    node6_cmdb_earliest = pd.NA

# 4) Top fault run in productcatalog_faults by max_relative_severity
productcatalog_faults['max_relative_severity'] = pd.to_numeric(productcatalog_faults['max_relative_severity'], errors='coerce')
if not productcatalog_faults.empty:
    productcatalog_top_fault = productcatalog_faults.sort_values('max_relative_severity', ascending=False, na_position='last').head(1)[
        ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity']
    ].copy()
    productcatalog_top_fault['max_relative_severity'] = productcatalog_top_fault['max_relative_severity'].round(6)
    productcatalog_top_fault['run_start_ts'] = productcatalog_top_fault['run_start_ts'].astype('Int64')
    productcatalog_top_fault['run_end_ts'] = productcatalog_top_fault['run_end_ts'].astype('Int64')
else:
    productcatalog_top_fault = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity'])

# Save filtered subsets in kernel variables for follow-up
# node6_kpi_faults, node6_cmdb_faults, productcatalog_top_fault already defined above

# Return only the compact aggregates as requested
node6_kpi_faults_display, node6_kpi_count, node6_kpi_earliest, node6_cmdb_faults_display, node6_cmdb_count, node6_cmdb_earliest, productcatalog_top_fault

```
Out[9]:
```
```python
# Plain-English compact summary of node-6 and productcatalogservice severe fault runs
summary = (
    "Node-6 and productcatalogservice fault-run summary (compact):\n\n"
    "- KPI entries containing 'node-6':\n"
    "  - Total fault runs: 3,394\n"
    "  - Earliest run_start_ts: 1647790200\n"
    "  - Top examples (by max_relative_severity):\n"
    "    * paymentservice-2 — container.node-6.container_cpu_system_seconds, 1647790200–1647792000, 31 min, mixed, max_rel_sev=1.1259e+13\n"
    "    * productcatalogservice-0 — container.node-6.container_fs_reads./dev/vda, 1647790200–1647792000, 31 min, mixed, max_rel_sev=632.0\n"
    "    * productcatalogservice-0 — container.node-6.container_fs_reads_MB./dev/vda, 1647790200–1647792000, 31 min, mixed, max_rel_sev≈135.61\n"
    "    * Multiple services show short high-severity network_receive anomalies (single-minute or few-minute runs, severities ~21→5).\n\n"
    "- Node-level faults where cmdb_id == 'node-6':\n"
    "  - Count: 137 runs\n"
    "  - Earliest run_start_ts: 1647790200\n"
    "  - Top node metrics (by severity) include system.load.1 (high), system.io.w_await, and network bytes/packets anomalies (severities up to ~3.10 for load and lower for network metrics).\n\n"
    "- Productcatalogservice top fault run:\n"
    "  - productcatalogservice-0 — container.node-6.container_fs_reads./dev/vda, 1647790200–1647792000, 31 min, mixed, max_rel_sev=632.0\n\n"
    "Interpretation / next steps:\n"
    "- Very large container filesystem-read anomalies (on node-6, affecting productcatalogservice pods) and extreme CPU-system metric on paymentservice-2 are standout signals.\n"
    "- Both many pod-level container KPIs and node-level system I/O / load metrics point to node-6 as a hotspot; productcatalogservice pods on node-6 show the single largest container fs read severity.\n"
    "- Recommend further investigation on node-6 disk/io subsystem and the containers’ filesystem activity for root-cause analysis. The filtered subsets node6_kpi_faults, node6_cmdb_faults, and productcatalog_top_fault are saved in the kernel for follow-up."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                           kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity
7801          paymentservice-2      container.node-6.container_cpu_system_seconds    1647790200  1647792000                  31                mixed           1.125900e+13
8113   productcatalogservice-0       container.node-6.container_fs_reads./dev/vda    1647790200  1647792000                  31                mixed           6.320000e+02
8115   productcatalogservice-0    container.node-6.container_fs_reads_MB./dev/vda    1647790200  1647792000                  31                mixed           1.356087e+02
7593          paymentservice-0  container.node-6.container_network_receive_MB....    1647790680  1647790680                   1                 high           2.181847e+01
4587            emailservice-0  container.node-6.container_network_receive_MB....    1647791460  1647791460                   1                 high           2.098265e+01
5088           emailservice2-0  container.node-6.container_network_receive_MB....    1647791160  1647791160                   1                 high           1.775518e+01
459                adservice-0  container.node-6.container_network_receive_MB....    1647791880  1647791880                   1                 high           1.294246e+01
2682         checkoutservice-1  container.node-6.container_network_receive_MB....    1647791400  1647791400                   1                 high           1.133248e+01
7729          paymentservice-1  container.node-6.container_network_receive_MB....    1647790680  1647790740                   2                 high           1.119632e+01
2243         checkoutservice-0  container.node-6.container_network_receive_MB....    1647791100  1647791160                   2                mixed           1.098630e+01
1844             cartservice-2  container.node-6.container_network_receive_MB....    1647790740  1647790800                   2                mixed           1.071783e+01
8872  productcatalogservice2-0  container.node-6.container_memory_failures.hie...    1647790200  1647792000                  31                mixed           1.000000e+01
8867  productcatalogservice2-0  container.node-6.container_memory_failures.con...    1647790200  1647792000                  31                mixed           1.000000e+01
4935            emailservice-2  container.node-6.container_network_receive_MB....    1647791760  1647791820                   2                 high           9.551643e+00
4755            emailservice-1  container.node-6.container_network_receive_MB....    1647791580  1647791700                   3                mixed           9.525499e+00
9714             redis-cart2-0  container.node-6.container_network_receive_MB....    1647791220  1647791280                   2                 high           7.023970e+00
9397   recommendationservice-2  container.node-6.container_network_receive_MB....    1647791880  1647791880                   1                 high           6.451129e+00
9225   recommendationservice-1  container.node-6.container_network_receive_MB....    1647790500  1647790500                   1                 high           5.868475e+00
1661             cartservice-1  container.node-6.container_network_receive_MB....    1647791280  1647791340                   2                 high           5.492821e+00
1489             cartservice-0  container.node-6.container_network_receive_MB....    1647791280  1647791340                   2                 high           5.314875e+00, 3394, 1647790200,      cmdb_id                     kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity
7453  node-6                system.load.1    1647790500  1647790620                   3                 high               3.104585
7445  node-6            system.io.w_await    1647792000  1647792000                   1                 high               1.920792
7467  node-6        system.net.bytes_rcvd    1647792000  1647792000                   1                 high               0.485753
7470  node-6        system.net.bytes_sent    1647792000  1647792000                   1                 high               0.483726
7455  node-6                system.load.5    1647790500  1647790680                   4                 high               0.454887
7473  node-6  system.net.packets_in.count    1647792000  1647792000                   1                 high               0.418797
7487  node-6  system.net.udp.in_datagrams    1647792000  1647792000                   1                 high               0.379333
7469  node-6        system.net.bytes_sent    1647791700  1647791820                   3                 high               0.360529
7466  node-6        system.net.bytes_rcvd    1647791700  1647791820                   3                 high               0.355689
7472  node-6  system.net.packets_in.count    1647791700  1647791820                   3                 high               0.306132, 137, 1647790200,                       cmdb_id                                      kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity
8113  productcatalogservice-0  container.node-6.container_fs_reads./dev/vda    1647790200  1647792000                  31                mixed                  632.0)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np

# Reuse kernel variables: faults_df, productcatalog_faults, productcatalog_top_fault (if exists)

# 1) Node-level faults where cmdb_id == 'node-6'
node6_node_faults = faults_df[faults_df['cmdb_id'] == 'node-6'].copy()

# Summary: count, earliest run_start_ts, max max_relative_severity
node6_node_count = int(node6_node_faults.shape[0])
node6_node_earliest = (int(node6_node_faults['run_start_ts'].min())
                       if node6_node_count > 0 and node6_node_faults['run_start_ts'].notna().any()
                       else pd.NA)
node6_node_max_sev = (float(node6_node_faults['max_relative_severity'].max())
                      if node6_node_count > 0 and node6_node_faults['max_relative_severity'].notna().any()
                      else np.nan)

node6_node_summary = pd.DataFrame([{
    'count_of_runs': node6_node_count,
    'earliest_run_start_ts': node6_node_earliest,
    'max_max_relative_severity': round(node6_node_max_sev, 6) if not np.isnan(node6_node_max_sev) else np.nan
}])

# Top 5 node-level kpi_names by max_relative_severity
if not node6_node_faults.empty:
    node6_node_top_kpis = (
        node6_node_faults
        .groupby('kpi_name', as_index=False)
        .agg(max_relative_severity=('max_relative_severity','max'))
        .sort_values('max_relative_severity', ascending=False)
        .head(5)
    )
    node6_node_top_kpis['max_relative_severity'] = node6_node_top_kpis['max_relative_severity'].round(6)
else:
    node6_node_top_kpis = pd.DataFrame(columns=['kpi_name','max_relative_severity'])

# 2) Container-level faults tied to node-6: kpi_name contains 'node-6' AND cmdb_id != 'node-6'
node6_container_faults = faults_df[
    faults_df['kpi_name'].astype(str).str.contains('node-6', case=True, na=False) &
    (faults_df['cmdb_id'] != 'node-6')
].copy()

container_count = int(node6_container_faults.shape[0])
container_distinct_pods = int(node6_container_faults['cmdb_id'].nunique())
container_earliest = (int(node6_container_faults['run_start_ts'].min())
                      if container_count > 0 and node6_container_faults['run_start_ts'].notna().any()
                      else pd.NA)
container_max_sev = (float(node6_container_faults['max_relative_severity'].max())
                     if container_count > 0 and node6_container_faults['max_relative_severity'].notna().any()
                     else np.nan)

container_summary = pd.DataFrame([{
    'count_of_runs': container_count,
    'number_of_distinct_pods_affected': container_distinct_pods,
    'earliest_run_start_ts': container_earliest,
    'max_max_relative_severity': round(container_max_sev, 6) if not np.isnan(container_max_sev) else np.nan
}])

# 3) Top 10 affected pod cmdb_ids (pods whose KPIs contain 'node-6') ranked by number_of_fault_runs
if not node6_container_faults.empty:
    pod_grp = node6_container_faults.groupby('cmdb_id', as_index=False).agg(
        number_of_fault_runs=('kpi_name','size'),
        earliest_run_start_ts=('run_start_ts','min')
    )
    # For each pod, determine its top_kpi_by_max_severity and severity
    pod_top_kpis = (
        node6_container_faults
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(kpi_max_sev=('max_relative_severity','max'))
    )
    # select top kpi per pod by kpi_max_sev
    pod_top_kpi_each = pod_top_kpis.sort_values(['cmdb_id','kpi_max_sev'], ascending=[True,False]).groupby('cmdb_id').first().reset_index()
    pod_summary = pod_grp.merge(pod_top_kpi_each[['cmdb_id','kpi_name','kpi_max_sev']], on='cmdb_id', how='left')
    pod_summary.rename(columns={'kpi_name':'top_kpi_by_max_severity','kpi_max_sev':'top_kpi_max_relative_severity'}, inplace=True)
    pod_summary['top_kpi_max_relative_severity'] = pod_summary['top_kpi_max_relative_severity'].round(6)
    node6_top_pods = pod_summary.sort_values('number_of_fault_runs', ascending=False).head(10).reset_index(drop=True)
    node6_top_pods['earliest_run_start_ts'] = node6_top_pods['earliest_run_start_ts'].astype('Int64')
else:
    node6_top_pods = pd.DataFrame(columns=['cmdb_id','number_of_fault_runs','earliest_run_start_ts','top_kpi_by_max_severity','top_kpi_max_relative_severity'])

# 4) Return the single productcatalog_top_fault (already saved earlier). If not present, derive from productcatalog_faults
try:
    # productcatalog_top_fault may already be a DataFrame in kernel
    productcatalog_top_fault
except NameError:
    if not productcatalog_faults.empty:
        productcatalog_top_fault = productcatalog_faults.sort_values('max_relative_severity', ascending=False).head(1)[
            ['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity']
        ].copy()
        productcatalog_top_fault['max_relative_severity'] = productcatalog_top_fault['max_relative_severity'].round(6)
        productcatalog_top_fault['run_start_ts'] = productcatalog_top_fault['run_start_ts'].astype('Int64')
        productcatalog_top_fault['run_end_ts'] = productcatalog_top_fault['run_end_ts'].astype('Int64')
    else:
        productcatalog_top_fault = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_types_in_run','max_relative_severity'
        ])

# Save filtered subsets in kernel variables for follow-up (already defined)
node6_node_faults, node6_node_summary, node6_node_top_kpis, node6_container_faults, container_summary, node6_top_pods, productcatalog_top_fault

```
Out[10]:
```
```python
# Plain-English summary string (compact) based on the kernel query results.
summary = (
    "Node-6 vs container-level severity (compact):\n\n"
    "1) Node-level faults (cmdb_id == 'node-6'):\n"
    f"   - count_of_runs: 137\n"
    f"   - earliest_run_start_ts: 1647790200\n"
    f"   - max(max_relative_severity): 3.104585\n"
    "   - top node KPIs by severity: system.load.1 (3.104585), system.io.w_await (1.920792),\n"
    "     system.net.bytes_rcvd (0.485753), system.net.bytes_sent (0.483726), system.load.5 (0.454887).\n\n"
    "2) Container-level faults tied to node-6 (kpi_name contains 'node-6', cmdb_id != 'node-6'):\n"
    f"   - count_of_runs: 3,394\n"
    f"   - number_of_distinct_pods_affected: 36\n"
    f"   - earliest_run_start_ts: 1647790200\n"
    f"   - max(max_relative_severity): 1.1259e+13 (extreme value from paymentservice-2 container metric)\n\n"
    "3) Top affected pods (pods whose KPIs contain 'node-6') — top examples by number_of_fault_runs:\n"
    "   - emailservice-2: 112 runs, earliest 1647790200, top_kpi container.node-6.container_network_receive_MB.... (max_rel_sev ≈ 9.551643)\n"
    "   - checkoutservice-0: 111 runs, earliest 1647790200, top_kpi container.node-6.container_network_receive_MB.... (≈11.9863)\n"
    "   - productcatalogservice2-0: 111 runs, earliest 1647790200, top_kpi container.node-6.container_memory_failures... (10.0)\n"
    "   - currencyservice2-0, paymentservice-2, currencyservice-0, cartservice-1, cartservice-0, productcatalogservice-1, currencyservice-2 (each in top-10 list).\n\n"
    "4) Top productcatalogservice fault (confirming earlier result):\n"
    "   - productcatalogservice-0, container.node-6.container_fs_reads./dev/vda,\n"
    "     run_start_ts=1647790200, run_end_ts=1647792000, length=31 min, mixed, max_relative_severity=632.0\n\n"
    "Interpretation / recommendation:\n"
    "- Node-6 is a clear hotspot: both node-level I/O/load anomalies and many container-level anomalies referencing 'node-6' started at the same early timestamp (1647790200).\n"
    "- Container filesystem-read anomalies on node-6 (productcatalogservice pods) show very high severity (fs_reads ./dev/vda, severity=632.0).\n"
    "- An extreme outlier (paymentservice-2 container CPU-system metric) shows an astronomic severity (≈1.1259e+13) and should be validated (instrumentation/normalization might be involved).\n"
    "- Actionable next steps: investigate node-6 disk/io subsystem and container filesystem activity (especially productcatalogservice pods), validate the extreme paymentservice-2 metric, and correlate with logs/trace timelines.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity  extremal_value  high_count  low_count service
7388  node-6        ping.can_connect    1647790200  1647792000                  31                mixed               0.000000             NaN          31         31  node-6
7389  node-6       system.cpu.iowait    1647790200  1647790200                   1                  low               0.003891           12.80           0          1  node-6
7390  node-6       system.cpu.iowait    1647790320  1647790320                   1                  low               0.015564           12.65           0          1  node-6
7391  node-6       system.cpu.iowait    1647790620  1647790740                   3                mixed               0.104025           12.44           1          2  node-6
7392  node-6       system.cpu.iowait    1647791580  1647791580                   1                 high               0.253870           20.25           1          0  node-6
...      ...                     ...           ...         ...                 ...                  ...                    ...             ...         ...        ...     ...
7520  node-6  system.udp.connect.num    1647790920  1647790920                   1                 high               0.000000           15.00           1          0  node-6
7521  node-6  system.udp.connect.num    1647791100  1647791100                   1                 high               0.000000           15.00           1          0  node-6
7522  node-6  system.udp.connect.num    1647791220  1647791400                   4                 high               0.000000           15.00           4          0  node-6
7523  node-6  system.udp.connect.num    1647791580  1647791640                   2                 high               0.000000           15.00           2          0  node-6
7524  node-6  system.udp.connect.num    1647791760  1647791940                   4                mixed               0.000000           15.00           3          1  node-6

[137 rows x 11 columns],    count_of_runs  earliest_run_start_ts  max_max_relative_severity
0            137             1647790200                   3.104585,                  kpi_name  max_relative_severity
23          system.load.1               3.104585
21      system.io.w_await               1.920792
33  system.net.bytes_rcvd               0.485753
34  system.net.bytes_sent               0.483726
25          system.load.5               0.454887,                   cmdb_id                                           kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity  extremal_value  high_count  low_count           service
394           adservice-0         container.node-6.container_cpu_cfs_periods    1647790740  1647790740                   1                  low               0.128713            88.0           0          1         adservice
395           adservice-0         container.node-6.container_cpu_cfs_periods    1647790920  1647791040                   3                mixed               0.054455            95.5           1          2         adservice
396           adservice-0         container.node-6.container_cpu_cfs_periods    1647791640  1647791640                   1                 high               0.000000           134.0           1          0         adservice
397           adservice-0         container.node-6.container_cpu_cfs_periods    1647791940  1647791940                   1                 high               0.048507           140.5           1          0         adservice
398           adservice-0  container.node-6.container_cpu_cfs_throttled_p...    1647790200  1647792000                  31                mixed                    NaN             NaN          31         31         adservice
...                   ...                                                ...           ...         ...                 ...                  ...                    ...             ...         ...        ...               ...
10352  shippingservice2-0     container.node-6.container_tasks_state.stopped    1647790200  1647792000                  31                mixed                    NaN             NaN          31         31  shippingservice2
10353  shippingservice2-0  container.node-6.container_tasks_state.uninter...    1647790200  1647792000                  31                mixed                    NaN             NaN          31         31  shippingservice2
10354  shippingservice2-0                 container.node-6.container_threads    1647790200  1647792000                  31                mixed               0.000000             NaN          31         31  shippingservice2
10355  shippingservice2-0             container.node-6.container_threads_max    1647790200  1647792000                  31                mixed                    NaN             NaN          31         31  shippingservice2
10356  shippingservice2-0  container.node-6.container_ulimits_soft.max_op...    1647790200  1647792000                  31                mixed               0.000000             NaN          31         31  shippingservice2

[3394 rows x 11 columns],    count_of_runs  number_of_distinct_pods_affected  earliest_run_start_ts  max_max_relative_severity
0           3394                                36             1647790200               1.125900e+13,                     cmdb_id  number_of_fault_runs  earliest_run_start_ts                            top_kpi_by_max_severity  top_kpi_max_relative_severity
0            emailservice-2                   112             1647790200  container.node-6.container_network_receive_MB....                   9.551643e+00
1         checkoutservice-0                   111             1647790200  container.node-6.container_network_receive_MB....                   1.098630e+01
2  productcatalogservice2-0                   111             1647790200  container.node-6.container_memory_failures.con...                   1.000000e+01
3        currencyservice2-0                   108             1647790200  container.node-6.container_cpu_cfs_throttled_p...                   1.000000e+00
4          paymentservice-2                   107             1647790200      container.node-6.container_cpu_system_seconds                   1.125900e+13
5         currencyservice-0                   107             1647790200  container.node-6.container_cpu_cfs_throttled_s...                   4.337278e+00
6             cartservice-1                   105             1647790200  container.node-6.container_network_receive_MB....                   5.492821e+00
7             cartservice-0                   105             1647790200  container.node-6.container_network_receive_MB....                   5.314875e+00
8   productcatalogservice-1                   104             1647790200  container.node-6.container_memory_failures.con...                   2.376289e+00
9         currencyservice-2                   104             1647790200  container.node-6.container_network_receive_MB....                   8.550290e-01,                       cmdb_id                                      kpi_name  run_start_ts  run_end_ts  run_length_minutes anomaly_types_in_run  max_relative_severity
8113  productcatalogservice-0  container.node-6.container_fs_reads./dev/vda    1647790200  1647792000                  31                mixed                  632.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import re

# Reuse existing kernel variable node6_container_faults
# If not present, fall back to selecting from faults_df
try:
    node6_container_faults
except NameError:
    node6_container_faults = faults_df[
        faults_df['kpi_name'].astype(str).str.contains('node-6', case=True, na=False) &
        (faults_df['cmdb_id'] != 'node-6')
    ].copy()

# Ensure numeric severity
node6_container_faults['max_relative_severity'] = pd.to_numeric(node6_container_faults['max_relative_severity'], errors='coerce')

# 1) Aggregate per pod
pods_agg = node6_container_faults.groupby('cmdb_id', as_index=False).agg(
    number_of_fault_runs=('kpi_name', 'size'),
    earliest_run_start_ts=('run_start_ts', 'min')
)

# 2) Per-pod top KPI by max_relative_severity
kpi_max = (
    node6_container_faults
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(kpi_max_relative_severity=('max_relative_severity','max'))
)

# pick top kpi per pod by kpi_max_relative_severity (descending)
kpi_top_per_pod = (
    kpi_max
    .sort_values(['cmdb_id','kpi_max_relative_severity'], ascending=[True, False])
    .drop_duplicates(subset=['cmdb_id'], keep='first')
    .rename(columns={'kpi_name':'top_kpi_by_max_severity','kpi_max_relative_severity':'top_kpi_max_relative_severity'})
)

# Merge pods_agg with top KPI info
node6_affected_pods = pods_agg.merge(kpi_top_per_pod[['cmdb_id','top_kpi_by_max_severity','top_kpi_max_relative_severity']], on='cmdb_id', how='left')

# Normalize service name by stripping trailing -<num> (keep node-* as-is if present)
def normalize_service(cmdb):
    if pd.isna(cmdb):
        return cmdb
    s = str(cmdb)
    m = re.match(r'^(.+?)-\d+$', s)
    if m:
        return m.group(1)
    return s

node6_affected_pods['normalized_service_name'] = node6_affected_pods['cmdb_id'].apply(normalize_service)

# Format numeric columns compactly
node6_affected_pods['top_kpi_max_relative_severity'] = pd.to_numeric(node6_affected_pods['top_kpi_max_relative_severity'], errors='coerce').round(6)
node6_affected_pods['earliest_run_start_ts'] = node6_affected_pods['earliest_run_start_ts'].astype('Int64')

# Sort by number_of_fault_runs descending and limit to top 20
node6_affected_pods = node6_affected_pods.sort_values('number_of_fault_runs', ascending=False).head(20).reset_index(drop=True)

# 3) Total number of distinct services affected
total_number_of_distinct_services_affected = int(node6_affected_pods['normalized_service_name'].nunique())

# Save node6_affected_pods in kernel (already assigned) and return compact results
node6_affected_pods[['cmdb_id','normalized_service_name','number_of_fault_runs','earliest_run_start_ts','top_kpi_by_max_severity','top_kpi_max_relative_severity']], total_number_of_distinct_services_affected

```
Out[11]:
```
Summary (plain English):

- Found 20 top pods whose KPIs reference "node-6" (limited view shown). These pods come from 12 distinct normalized services in total.

- Top affected pods by number of fault runs:
  1) emailservice-2 — 112 runs; earliest 1647790200; top KPI: container.node-6.container_network_receive_MB..., max_rel_sev ≈ 9.55  
  2) checkoutservice-0 — 111 runs; earliest 1647790200; top KPI: container.node-6.container_network_receive_MB..., max_rel_sev ≈ 10.99  
  3) productcatalogservice2-0 — 111 runs; earliest 1647790200; top KPI: container.node-6.container_memory_failures..., max_rel_sev = 10.0  
  4) currencyservice2-0 — 108 runs; earliest 1647790200; top KPI: throttling/CPU-related, max_rel_sev = 1.0  
  5) paymentservice-2 — 107 runs; earliest 1647790200; top KPI: container.node-6.container_cpu_system_seconds, extreme max_rel_sev ≈ 1.1259e+13 (outlier)

- productcatalogservice pods appear among the top affected:
  - productcatalogservice-0: 99 runs; top KPI container.node-6.container_fs_reads./dev/vda with max_rel_sev = 632.0
  - productcatalogservice-1 and -2 also in top list (104 and 102 runs respectively), with memory-failure and fs-read KPIs showing elevated severities.

- Earliest anomaly time for these pods is consistent: most earliest_run_start_ts = 1647790200 (2022-03-20 23:30:00 UTC+8).

- Common top KPIs across pods reference container network receive, container CPU/system, container memory failures, and container fs_reads — indicating node-6 I/O/network and container resource issues.

- Notable observations:
  - paymentservice-2 shows an extreme outlier severity (1.1259e+13) — validate instrumentation/normalization for this metric.
  - productcatalogservice-0 shows very high filesystem-read severity (632.0) on node-6 — a likely real hotspot.
  - Many services (email, checkout, cart, currency, productcatalog, adservice, paymentservice, etc.) show high counts and early timestamps pointing to a correlated issue centered on node-6.

- Next recommended actions: prioritize investigating node-6 disk I/O and container filesystem activity (especially fs_reads on productcatalogservice pods) and validate the paymentservice-2 CPU metric anomaly.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id normalized_service_name  number_of_fault_runs  earliest_run_start_ts                            top_kpi_by_max_severity  top_kpi_max_relative_severity
0             emailservice-2            emailservice                   112             1647790200  container.node-6.container_network_receive_MB....                   9.551643e+00
1          checkoutservice-0         checkoutservice                   111             1647790200  container.node-6.container_network_receive_MB....                   1.098630e+01
2   productcatalogservice2-0  productcatalogservice2                   111             1647790200  container.node-6.container_memory_failures.con...                   1.000000e+01
3         currencyservice2-0        currencyservice2                   108             1647790200  container.node-6.container_cpu_cfs_throttled_p...                   1.000000e+00
4           paymentservice-2          paymentservice                   107             1647790200      container.node-6.container_cpu_system_seconds                   1.125900e+13
5          currencyservice-0         currencyservice                   107             1647790200  container.node-6.container_cpu_cfs_throttled_s...                   4.337278e+00
6              cartservice-1             cartservice                   105             1647790200  container.node-6.container_network_receive_MB....                   5.492821e+00
7              cartservice-0             cartservice                   105             1647790200  container.node-6.container_network_receive_MB....                   5.314875e+00
8    productcatalogservice-1   productcatalogservice                   104             1647790200  container.node-6.container_memory_failures.con...                   2.376289e+00
9          currencyservice-2         currencyservice                   104             1647790200  container.node-6.container_network_receive_MB....                   8.550290e-01
10   productcatalogservice-2   productcatalogservice                   102             1647790200    container.node-6.container_fs_reads_MB./dev/vda                   3.335607e+00
11               adservice-1               adservice                   101             1647790200      container.node-6.container_cpu_system_seconds                   5.000000e-01
12        checkoutservice2-0        checkoutservice2                   100             1647790200         container.node-6.container_cpu_cfs_periods                   1.000000e+00
13             cartservice-2             cartservice                   100             1647790200  container.node-6.container_network_receive_MB....                   1.071783e+01
14           emailservice2-0           emailservice2                    99             1647790200  container.node-6.container_network_receive_MB....                   1.775518e+01
15               adservice-0               adservice                    99             1647790200  container.node-6.container_network_receive_MB....                   1.294246e+01
16   productcatalogservice-0   productcatalogservice                    99             1647790200       container.node-6.container_fs_reads./dev/vda                   6.320000e+02
17         currencyservice-1         currencyservice                    99             1647790200      container.node-6.container_cpu_system_seconds                   5.000000e-01
18          paymentservice-1          paymentservice                    99             1647790200  container.node-6.container_network_receive_MB....                   1.119632e+01
19        shippingservice2-0        shippingservice2                    98             1647790200         container.node-6.container_cpu_cfs_periods                   3.333330e-01, 12)```
```